# 1. Install AG2 + Materials Project dependencies

In [11]:
!pip install -U "ag2[openai]" autogen -q

import sys
!{sys.executable} -m pip install pymatgen mp-api numpy cython ipywidgets jupyterlab_widgets -q

print("✓ AG2/autogen + Pymatgen + MP-API installed.")

✓ AG2/autogen + Pymatgen + MP-API installed.


# 2. Imports + LLM Configuration


In [12]:
import subprocess, shutil

if shutil.which("tectonic") is None:
    subprocess.run(
        ["conda", "install", "-c", "conda-forge", "tectonic", "-y", "--quiet"],
        check=False
    )

if shutil.which("tectonic") is not None:
    print("\u2713 Tectonic (LaTeX compiler) ready.")
else:
    print("\u26a0\ufe0f  Tectonic not found. Run manually: conda install -c conda-forge tectonic")

import os
import json
import random
import base64
import re
from datetime import datetime, timedelta, timezone
from typing import Annotated, Any, Literal, Union, List, Optional, Tuple
from enum import Enum
from pathlib import Path

from openai import OpenAI

from autogen import ConversableAgent
from autogen.agentchat import ReplyResult
from autogen.agentchat.group import (
    ContextVariables,
    AgentTarget, AgentNameTarget, StayTarget,
    OnCondition, StringLLMCondition,
    OnContextCondition, ExpressionContextCondition, ContextExpression,
    RevertToUserTarget, TerminateTarget
)
from autogen.agentchat.groupchat import GroupChat, GroupChatManager
from autogen.tools import tool

from mp_api.client import MPRester

from autogen.coding.local_commandline_code_executor import LocalCommandLineCodeExecutor
from autogen.coding.base import CodeBlock
from pydantic import BaseModel, Field

from autogen.agentchat.group.patterns import DefaultPattern
from autogen.agentchat import initiate_group_chat



# --- LLM CONFIGURATION


client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

llm_config = {
    "model": "gpt-4o",
    "api_key": os.environ["OPENAI_API_KEY"],
}


def llm_completion(
    *,
    client: OpenAI,
    model: str,
    input: list,
    text_format=None,
    max_tokens: int = 2000,
    temperature: float = 0.2,
):
    if not isinstance(model, str) or not model.strip():
        raise ValueError("Model name must be a non-empty string.")

    model_name = model.strip()
    reasoning_model = model_name.startswith("o")
    

    # ---- PARSED OUTPUT MODE ----
    
    if text_format is not None:

        kwargs = dict(
            model=model_name,
            input=input,
            text_format=text_format,
            max_output_tokens=max_tokens,
        )

        if not reasoning_model:
            kwargs["temperature"] = temperature

        resp = client.responses.parse(**kwargs)
        return resp.output_parsed

    # ---- NORMAL OUTPUT MODE ----

    kwargs = dict(
        model=model_name,
        input=input,
        max_output_tokens=max_tokens,
    )

    if not reasoning_model:
        kwargs["temperature"] = temperature

    resp = client.responses.create(**kwargs)

    return getattr(resp, "output_text", None)



# ---PROJECT FOLDER 


PROJECT_FOLDER = os.path.abspath("ag2_project")
os.makedirs(PROJECT_FOLDER, exist_ok=True)
os.environ["PROJECT_FOLDER"] = PROJECT_FOLDER

PLOTS_V1_DIR = os.path.join(PROJECT_FOLDER, "plots")
PLOTS_V2_DIR = os.path.join(PROJECT_FOLDER, "plots_v2")

os.makedirs(PLOTS_V1_DIR, exist_ok=True)
os.makedirs(PLOTS_V2_DIR, exist_ok=True)

print("✓ Imports loaded and LLM configured.")

✓ Tectonic (LaTeX compiler) ready.
✓ Imports loaded and LLM configured.


# 3. Data Contracts — (Plot Artifacts + Visual Validation, Writer)

In [13]:
# --- Plot Contracts ---

class PlotType(str, Enum):
    scatter_2d = "scatter_2d"
    histogram = "histogram"
    bar = "bar"
    heatmap_2d = "heatmap_2d"
    scatter_3d = "scatter_3d"
    pie = "pie"


class PlotAxes(BaseModel):
    type: Literal["2d", "3d", "hist", "heatmap", "categorical", "pie"] = Field(...)
    x: Optional[str] = None
    y: Optional[str] = None
    z: Optional[str] = None
    intensity: Optional[str] = None


class PlotCandidate(BaseModel):
    plot_id: str = Field(..., min_length=1)
    title: str = Field(..., min_length=1)
    plot_type: PlotType
    data_requirements: List[str]
    axes: PlotAxes

    def validate_structure(self):
        if self.plot_type == PlotType.scatter_2d:
            if self.axes.type != "2d":
                raise ValueError("scatter_2d requires axes.type='2d'")
            if not self.axes.x or not self.axes.y:
                raise ValueError("scatter_2d requires axes.x and axes.y")
            if self.axes.z or self.axes.intensity:
                raise ValueError("scatter_2d cannot include z or intensity")

        if self.plot_type == PlotType.histogram:
            if self.axes.type != "hist":
                raise ValueError("histogram requires axes.type='hist'")
            if not self.axes.x:
                raise ValueError("histogram requires axes.x")
            if self.axes.y or self.axes.z or self.axes.intensity:
                raise ValueError("histogram only supports axes.x")

        if self.plot_type == PlotType.bar:
            if self.axes.type != "categorical":
                raise ValueError("bar requires axes.type='categorical'")
            if not self.axes.x or not self.axes.y:
                raise ValueError("bar requires axes.x and axes.y")
            if self.axes.z or self.axes.intensity:
                raise ValueError("bar cannot include z or intensity")

        if self.plot_type == PlotType.heatmap_2d:
            if self.axes.type != "heatmap":
                raise ValueError("heatmap_2d requires axes.type='heatmap'")
            if not self.axes.x or not self.axes.y or not self.axes.intensity:
                raise ValueError("heatmap_2d requires axes.x, axes.y and axes.intensity")
            if self.axes.z:
                raise ValueError("heatmap_2d cannot include z")

        if self.plot_type == PlotType.scatter_3d:
            if self.axes.type != "3d":
                raise ValueError("scatter_3d requires axes.type='3d'")
            if not self.axes.x or not self.axes.y or not self.axes.z:
                raise ValueError("scatter_3d requires axes.x, axes.y and axes.z")
            if self.axes.intensity:
                raise ValueError("scatter_3d cannot include intensity")

        if self.plot_type == PlotType.pie:
            if self.axes.type != "pie":
                raise ValueError("pie requires axes.type='pie'")
            if not self.axes.x or not self.axes.y:
                raise ValueError("pie requires axes.x and axes.y")
            if self.axes.z or self.axes.intensity:
                raise ValueError("pie cannot include z or intensity")


# --- Plot Artifacts ---

class PlotArtifactMeta(BaseModel):
    plot_id: str = Field(..., min_length=1)
    name: str = Field(..., min_length=1)
    path: str = Field(..., min_length=1)
    description: str = Field(..., min_length=1)


def validate_plot_artifact_meta(item: dict) -> dict:
    meta = PlotArtifactMeta(**item)

    if not meta.name.startswith(f"{meta.plot_id}_"):
        raise ValueError("name must start with plot_id + '_'")

    if os.path.isabs(meta.path):
        raise ValueError("path must be relative")

    if not (meta.path.startswith("plots/") or meta.path.startswith("plots_v2/")):
        raise ValueError("path must start with 'plots/' or 'plots_v2/'")

    return meta.model_dump()


# --- Visual Validation ---

class PlotValidationResult(BaseModel):
    summary: str
    issues: List[str]
    improvements: List[str]
    severity: Literal["low", "medium", "high"]
    pass_fail: bool


def evaluate_plot(
    plot_name: str,
    plot_description: str,
    model: str,
    plot_path: str,
) -> PlotValidationResult:
    import base64

    if not isinstance(plot_name, str) or not plot_name.strip():
        raise ValueError("plot_name must be a non-empty string.")

    if not isinstance(plot_description, str) or not plot_description.strip():
        raise ValueError("plot_description must be a non-empty string.")

    if not isinstance(model, str) or not model.strip():
        raise ValueError("model must be a non-empty string.")

    if not isinstance(plot_path, str) or not plot_path.strip():
        raise ValueError("plot_path must be a non-empty string.")

    rel_path = plot_path.strip().replace("\\", "/")
    if os.path.isabs(rel_path) or ".." in rel_path.split("/"):
        raise ValueError("plot_path must be a safe relative path.")

    input_image = os.path.join(PROJECT_FOLDER, rel_path)

    if not os.path.exists(input_image):
        raise RuntimeError(f"Plot image not found at '{input_image}'")

    try:
        with open(input_image, "rb") as image_file:
            base64_image = base64.b64encode(image_file.read()).decode("utf-8")
    except Exception as e:
        raise RuntimeError(f"Failed to read plot image at '{input_image}': {e}")

    prompt = (
        "You are a strict plot reviewer.\n"
        "Provide qualitative visual feedback only.\n"
        "Focus on:\n"
        "- readability\n"
        "- axis labels\n"
        "- units\n"
        "- scaling\n"
        "- visual clarity\n"
        "Do NOT perform exact numeric reading.\n\n"
        f"Plot task description:\n{plot_description.strip()}"
    )

    parsed = llm_completion(
        client=client,
        model=model,
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_image",
                        "image_url": f"data:image/png;base64,{base64_image}",
                    },
                    {
                        "type": "input_text",
                        "text": prompt,
                    },
                ],
            }
        ],
        text_format=PlotValidationResult,
        max_tokens=10000,
    )

    if parsed is None:
        raise RuntimeError("LLM returned None during plot evaluation.")

    if not isinstance(parsed, PlotValidationResult):
        raise RuntimeError("LLM returned an invalid PlotValidationResult object.")

    return parsed


# --- Writer Contracts ---

class LatexSections(BaseModel):
    latex_sections: str = Field(..., min_length=1)


def validate_latex_sections(item: dict) -> dict:
    obj = LatexSections(**item)
    s = obj.latex_sections.strip()

    forbidden = [
        "\\documentclass",
        "\\usepackage",
        "\\begin{document}",
        "\\end{document}",
    ]
    for token in forbidden:
        if token in s:
            raise ValueError("latex_sections must not include LaTeX preamble or document environment.")

    if "\\begin{abstract}" not in s or "\\end{abstract}" not in s:
        raise ValueError("latex_sections must include an abstract environment.")

    return obj.model_dump()

# 4. User query + Context Variables

In [14]:
from autogen.agentchat.group import ContextVariables

context_variables = ContextVariables(
    data={
        "task_started": False,
        "query": None,

        "main_task_analysis": None,
        "main_task_improvements": None,
        "main_task_improved": None,

        "plan": None,
        "review_status": None,
        "review_notes": None,

        "search_criteria": None,
        "fields": None,
        "sample_number": None,
        "mp_results": None,

        "final_conclusion": None,
        "analysis_summary": None,
        "materials_data_path": None,

        "plot_candidates": [],
        "plot_selected": [],
        "plot_selection_notes": None,
        "plots_v1": [],
        "plot_reviews": {},
        "plots_v2": [],
        "plots_output_dir_v1": "plots",
        "plots_output_dir_v2": "plots_v2",
        "plots_path_v1": PLOTS_V1_DIR,
        "plots_path_v2": PLOTS_V2_DIR,

        "plot_suggest_status": "idle",
        "plot_select_status": "idle",
        "plot_code_status": "idle",
        "plot_validate_status": "idle",

        "image_artifacts_queue": [],
        "image_reviews": [],

        "last_executed_code": None,
        "last_execution_output": None,
        "last_execution_exit_code": None,
        "last_executed_file": None,

        "latex_sections": None,
        "report_sections_path": None,
        "report_tex_path": None,
        "report_tex_path_abs": None,
        "report_pdf_path": None,
        "latex_status": None,
        "latex_engine": None,
        "latex_passes": None,
        "latex_compile_output": None,
        "latex_log_path": None,
        "latex_log_tail": None,
        "latex_compile_error": None,

        "next_agent": None,
    }
)

context_variables["agent_registry"] = [
    "AgentTaskImprover",
    "AgentPlanner",
    "AgentPlanReviewer",
    "AgentMaterialsRetriever",
    "AgentAnalyzer",
    "AgentPlotSuggester",
    "AgentPlotSelector",
    "AgentCoder",
    "AgentPlotValidator",
    "AgentLatexWriter",
    "AgentLatexCompiler",
    "Human",
]

print("✓ ContextVariables initialized.")

✓ ContextVariables initialized.


# 5. LaTeX Template

In [15]:
LATEX_TEMPLATE = r"""
\documentclass{article}

\usepackage{graphicx}
\usepackage{float}
\usepackage{geometry}
\usepackage{caption}
\usepackage{booktabs}
\usepackage{hyperref}

\geometry{margin=1in}
\graphicspath{{plots/}{plots_v2/}}

\title{Automated Materials Screening Report}
\author{AG2 Multi-Agent System}
\date{\today}

\begin{document}
\maketitle

{section_content}

\end{document}
"""

# 6. TOOLS 

In [16]:
# Tool — task_improver

def task_improver_tool(
    analysis: Annotated[str, "Short analysis of the task consistency, missing info, redundancy."],
    improvements: Annotated[str, "Concrete improvements to make the task complete and non-redundant."],
    improved_task: Annotated[str, "Improved version of the main task."],
    context_variables: ContextVariables,
) -> ReplyResult:
    agent_name = "AgentTaskImprover"

    if context_variables.get("task_improver_status") == "completed":
        context_variables["next_agent"] = "AgentPlanner"
        return ReplyResult(
            message="task improver stage already completed",
            target=AgentNameTarget("AgentPlanner"),
            context_variables=context_variables,
        )

    context_variables["task_improver_status"] = "running"

    def _fail(message: str) -> ReplyResult:
        context_variables["task_improver_status"] = "failed"
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in task_improver_tool: {message}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    try:
        if not isinstance(analysis, str) or not analysis.strip():
            raise ValueError("analysis must be a non-empty string.")
        if not isinstance(improvements, str) or not improvements.strip():
            raise ValueError("improvements must be a non-empty string.")
        if not isinstance(improved_task, str) or not improved_task.strip():
            raise ValueError("improved_task must be a non-empty string.")

        original_query = context_variables.get("query", None)
        if original_query is not None:
            if not isinstance(original_query, str) or not original_query.strip():
                raise ValueError("context_variables['query'] must be a non-empty string when present.")

    except ValueError as e:
        return _fail(str(e))

    analysis_s = analysis.strip()
    improvements_s = improvements.strip()
    improved_task_s = improved_task.strip()
    original_query_s = (context_variables.get("query", "") or "").strip()

    lowered_task = improved_task_s.lower()
    lowered_query = original_query_s.lower()

    forbidden_added_requirements = [
        "png format",
        "pdf format",
        "best candidate",
        "top candidate",
        "rank the candidates",
        "ranking the candidates",
        "rank candidates",
        "multiple viable candidates",
        "comparison plot",
        "scatter plot",
        "histogram",
        "heatmap",
        "bar chart",
        "3d plot",
        "report in pdf",
        "scientific report",
        "or their compounds",
        "their compounds",
        "indirect compounds",
        "safety standards",
        "environmental compliance",
        "compliance",
        "real-world implications",
        "practical requirements",
    ]

    for token in forbidden_added_requirements:
        if token in lowered_task and token not in lowered_query:
            return _fail(
                "improved_task introduces extra requirements "
                f"that change the original scope ('{token}')."
            )

    query_mentions_plotting = any(
        token in lowered_query
        for token in ["plot", "plots", "plotting", "graph", "graphs", "chart", "charts", "visualization"]
    )

    task_mentions_over_specific_plotting = any(
        token in lowered_task
        for token in [
            "comparison plot",
            "scatter plot",
            "histogram",
            "heatmap",
            "bar chart",
            "3d plot",
            "plot relevant material properties",
            "plot of",
            "plotting step",
            "data visualization step",
        ]
    )

    if query_mentions_plotting and task_mentions_over_specific_plotting:
        generic_plot_request = not any(
            token in lowered_query
            for token in [
                "comparison plot",
                "scatter plot",
                "histogram",
                "heatmap",
                "bar chart",
                "3d plot",
            ]
        )
        if generic_plot_request:
            return _fail(
                "improved_task makes plotting more specific than the original query."
            )

    preserved_exact_phrases = [
        "stable",
        "include plotting",
    ]

    for phrase in preserved_exact_phrases:
        if phrase in lowered_query and phrase not in lowered_task:
            return _fail(
                f"improved_task removes or weakens an explicit user requirement ('{phrase}')."
            )

    import re

    def _extract_numeric_ranges(text: str):
        pattern = r"([a-zA-Z_]+)\s+(?:between|from)\s+([0-9]*\.?[0-9]+)\s+(?:and|to)\s+([0-9]*\.?[0-9]+)"
        ranges = {}
        for field, lo, hi in re.findall(pattern, text.lower()):
            ranges[field] = (lo, hi)
        return ranges

    query_ranges = _extract_numeric_ranges(lowered_query)
    task_ranges = _extract_numeric_ranges(lowered_task)

    for field, values in query_ranges.items():
        if field in task_ranges and task_ranges[field] != values:
            return _fail(
                f"improved_task changes the numeric range for '{field}' from {values} to {task_ranges[field]}."
            )

    explicit_elements = re.findall(r"\b[A-Z][a-z]?\b", original_query_s)
    toxic_elements = [x for x in explicit_elements if x in {"Pb", "Cd", "Hg"}]

    for element in toxic_elements:
        if element.lower() not in lowered_task:
            return _fail(
                f"improved_task removes the explicit excluded element '{element}'."
            )

    if "excluding toxic elements" in lowered_query and "excluding toxic elements" not in lowered_task:
        if "exclude materials containing toxic elements" not in lowered_task:
            return _fail(
                "improved_task weakens or alters the original exclusion wording."
            )

    normalized_query = re.sub(r"\s+", " ", lowered_query).strip()
    normalized_task = re.sub(r"\s+", " ", lowered_task).strip()

    suspicious_additions = [
        "suitable",
        "compliance",
        "environmental safety",
        "environmental harm",
        "insulation",
        "outdoor",
        "protects infrastructure",
        "high-voltage areas",
    ]

    for token in suspicious_additions:
        if token in normalized_task and token not in normalized_query:
            return _fail(
                f"improved_task adds inferred context that was not explicitly requested ('{token}')."
            )

    context_variables["task_started"] = True
    context_variables["main_task_analysis"] = analysis_s
    context_variables["main_task_improvements"] = improvements_s
    context_variables["main_task_improved"] = improved_task_s
    context_variables["task_improver_status"] = "completed"
    context_variables["next_agent"] = "AgentPlanner"

    message = {
        "main_task_analysis": analysis_s,
        "main_task_improvements": improvements_s,
        "main_task_improved": improved_task_s,
    }

    return ReplyResult(
        message=json.dumps(message, indent=2, ensure_ascii=False),
        target=AgentNameTarget("AgentPlanner"),
        context_variables=context_variables,
    )

# Tool — planner

def planner_tool(
    plan: Annotated[dict | None, "Structured plan dict generated by the Planner agent."],
    plan_description: Annotated[str | None, "Short description of the plan."],
    success_criteria: Annotated[List[str] | None, "Concrete criteria to consider the run successful."],
    plan_score: Annotated[float | None, "Planner self-score for the plan quality (0-10)."],
    context_variables: ContextVariables,
) -> ReplyResult:
    agent_name = "AgentPlanner"
    main_task_improved = context_variables.get("main_task_improved", None)

    if context_variables.get("planner_status") == "completed":
        existing_plan = context_variables.get("plan", None)
        existing_plan_description = context_variables.get("plan_description", None)
        existing_success_criteria = context_variables.get("success_criteria", None)
        existing_plan_score = context_variables.get("plan_score", None)

        next_agent_existing = "AgentPlanReviewer"
        if isinstance(existing_plan, dict):
            routing_existing = existing_plan.get("routing", {})
            if isinstance(routing_existing, dict):
                after_planner_existing = routing_existing.get("after_planner", None)
                if isinstance(after_planner_existing, str) and after_planner_existing.strip():
                    next_agent_existing = after_planner_existing.strip()

        context_variables["next_agent"] = next_agent_existing

        message = {
            "plan_description": existing_plan_description,
            "success_criteria": existing_success_criteria,
            "plan_score": existing_plan_score,
            "plan": existing_plan,
        }

        return ReplyResult(
            message=json.dumps(message, indent=2, ensure_ascii=False, default=str),
            target=AgentNameTarget(next_agent_existing),
            context_variables=context_variables,
        )

    context_variables["planner_status"] = "running"

    try:
        if not isinstance(main_task_improved, str) or not main_task_improved.strip():
            raise ValueError("context_variables['main_task_improved'] must be a non-empty string.")
        if not isinstance(plan, dict) or not plan:
            raise ValueError("plan must be a non-empty dict.")
        if not isinstance(plan_description, str) or not plan_description.strip():
            raise ValueError("plan_description must be a non-empty string.")
        if not isinstance(success_criteria, list) or not success_criteria or not all(
            isinstance(x, str) and x.strip() for x in success_criteria
        ):
            raise ValueError("success_criteria must be a non-empty list of non-empty strings.")
        if not isinstance(plan_score, (int, float)):
            raise ValueError("plan_score must be a number.")
        plan_score_f = float(plan_score)
        if plan_score_f < 0 or plan_score_f > 10:
            raise ValueError("plan_score must be between 0 and 10.")
    except ValueError as e:
        context_variables["planner_status"] = "failed"
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in planner_tool: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    main_task_s = main_task_improved.strip()

    try:
        if not isinstance(plan.get("schema_version"), str) or not plan["schema_version"].strip():
            raise ValueError("plan.schema_version must be a non-empty string.")

        if not isinstance(plan.get("main_task_improved"), str) or not plan["main_task_improved"].strip():
            raise ValueError("plan.main_task_improved must be a non-empty string.")
        if plan["main_task_improved"].strip() != main_task_s:
            raise ValueError("plan.main_task_improved must match context_variables['main_task_improved'].")

        routing = plan.get("routing")
        if not isinstance(routing, dict):
            raise ValueError("plan.routing must be a dict.")
        for k in (
            "after_planner",
            "on_approve",
            "on_revise",
            "on_handoff",
            "after_plot_suggest",
            "after_plot_select",
            "after_code",
            "after_plot_validate",
            "on_plot_validation_failed",
            "after_latex_write",
            "after_latex_compile",
            "on_latex_compile_failed",
        ):
            if not isinstance(routing.get(k), str) or not routing[k].strip():
                raise ValueError(f"plan.routing.{k} must be a non-empty string.")

        steps = plan.get("steps")
        if not isinstance(steps, list) or len(steps) == 0:
            raise ValueError("plan.steps must be a non-empty list.")
        if not all(isinstance(s, dict) for s in steps):
            raise ValueError("plan.steps must contain dict items only.")
        if not all(isinstance(s.get("id"), str) and s["id"].strip() for s in steps):
            raise ValueError("Each step must have a non-empty string 'id'.")

        retrieval = plan.get("retrieval")
        if not isinstance(retrieval, dict):
            raise ValueError("plan.retrieval must be a dict.")
        if not isinstance(retrieval.get("search_criteria"), dict):
            raise ValueError("plan.retrieval.search_criteria must be a dict.")

        search_criteria = retrieval.get("search_criteria")

        allowed_search_keys = {
            "band_gap",
            "energy_above_hull",
            "density",
            "num_sites",
            "k_voigt",
            "g_voigt",
            "elements",
            "chemsys",
            "exclude_elements",
        }

        found_keys = set(search_criteria.keys())
        invalid_keys = sorted(found_keys - allowed_search_keys)
        retrieval_errors = []

        if invalid_keys:
            retrieval_errors.append(
                f"plan.retrieval.search_criteria contains invalid keys: {invalid_keys}"
            )

        if "excluded_elements" in search_criteria:
            retrieval_errors.append(
                "plan.retrieval.search_criteria must use 'exclude_elements', not 'excluded_elements'."
            )

        numeric_range_keys = {
            "energy_above_hull",
            "band_gap",
            "density",
            "num_sites",
            "k_voigt",
            "g_voigt",
        }

        normalized_search_criteria = dict(search_criteria)

        for key in numeric_range_keys:
            if key in search_criteria:
                value = search_criteria[key]
                if not (
                    isinstance(value, (list, tuple))
                    and len(value) == 2
                    and all(isinstance(x, (int, float)) for x in value)
                ):
                    retrieval_errors.append(
                        f"plan.retrieval.search_criteria.{key} must be a list/tuple(min, max) of numbers."
                    )
                else:
                    min_v, max_v = value
                    if min_v > max_v:
                        retrieval_errors.append(
                            f"plan.retrieval.search_criteria.{key} must satisfy min <= max."
                        )
                    else:
                        normalized_search_criteria[key] = (float(min_v), float(max_v))

        if "exclude_elements" in search_criteria:
            exclude_elements = search_criteria["exclude_elements"]
            if not (
                isinstance(exclude_elements, list)
                and len(exclude_elements) > 0
                and all(isinstance(x, str) and x.strip() for x in exclude_elements)
            ):
                retrieval_errors.append(
                    "plan.retrieval.search_criteria.exclude_elements must be a non-empty list of non-empty strings."
                )
            else:
                normalized_search_criteria["exclude_elements"] = [x.strip() for x in exclude_elements]

        if retrieval_errors:
            raise ValueError(" | ".join(retrieval_errors))

        plan["retrieval"]["search_criteria"] = normalized_search_criteria

        if not isinstance(retrieval.get("fields"), list) or not retrieval["fields"] or not all(
            isinstance(x, str) and x.strip() for x in retrieval["fields"]
        ):
            raise ValueError("plan.retrieval.fields must be a non-empty list of non-empty strings.")

        allowed_fields = {
            "material_id",
            "formula_pretty",
            "band_gap",
            "energy_above_hull",
            "density",
            "volume",
            "symmetry",
            "nsites",
            "elements",
            "chemsys",
            "is_stable",
            "bulk_modulus",
            "shear_modulus",
        }

        invalid_fields = [x for x in retrieval["fields"] if x not in allowed_fields]
        if invalid_fields:
            raise ValueError(f"plan.retrieval.fields contains invalid values: {invalid_fields}")

        if not isinstance(retrieval.get("sample_number"), int) or retrieval["sample_number"] <= 0:
            raise ValueError("plan.retrieval.sample_number must be a positive int.")

        plotting = plan.get("plotting", None)
        if plotting is not None:
            if not isinstance(plotting, dict):
                raise ValueError("plan.plotting must be a dict if present.")
            enabled = plotting.get("enabled", None)
            if not isinstance(enabled, bool):
                raise ValueError("plan.plotting.enabled must be a bool.")
            if enabled is True:
                n_candidates = plotting.get("n_candidates", None)
                n_selected = plotting.get("n_selected", None)
                output_dir = plotting.get("output_dir", None)
                if not isinstance(n_candidates, int) or n_candidates != 10:
                    raise ValueError("plan.plotting.n_candidates must be int == 10.")
                if not isinstance(n_selected, int) or n_selected != 5:
                    raise ValueError("plan.plotting.n_selected must be int == 5.")
                if not isinstance(output_dir, str) or output_dir.strip() != "plots":
                    raise ValueError('plan.plotting.output_dir must be "plots".')

                plot_intents = plotting.get("plot_intents", None)
                if plot_intents is None:
                    raise ValueError("plan.plotting.plot_intents is required when plotting.enabled is True.")
                if not isinstance(plot_intents, list):
                    raise ValueError("plan.plotting.plot_intents must be a list[str].")
                if len(plot_intents) == 0:
                    raise ValueError("plan.plotting.plot_intents must be non-empty when plotting.enabled is True.")
                for i, intent in enumerate(plot_intents):
                    if not isinstance(intent, str):
                        raise ValueError(f"plan.plotting.plot_intents[{i}] must be a string.")
                    if not intent.strip():
                        raise ValueError(f"plan.plotting.plot_intents[{i}] must be a non-empty string.")

                retrieved_fields = {x.strip() for x in retrieval["fields"] if isinstance(x, str) and x.strip()}
                allowed_direct_plot_fields = set(retrieved_fields)
                allowed_derived_plot_fields = {"stability_bins", "band_gap_bins"}

                vague_tokens = {
                    "relevant properties",
                    "material suitability",
                    "suitability",
                    "stability metrics",
                    "material performance",
                    "performance metrics",
                    "relevant material properties",
                    "plot relevant properties",
                    "plot material suitability",
                    "plot stability metrics",
                    "plot material performance",
                }

                known_plot_words = {
                    "scatter",
                    "histogram",
                    "bar",
                    "heatmap",
                    "vs",
                    "and",
                    "of",
                    "by",
                    "with",
                    "for",
                    "count",
                    "counts",
                    "distribution",
                }

                known_plot_fields = allowed_direct_plot_fields | allowed_derived_plot_fields

                for i, intent in enumerate(plot_intents):
                    intent_s = intent.strip()
                    intent_l = intent_s.lower()

                    for token in vague_tokens:
                        if token in intent_l:
                            raise ValueError(
                                f"plan.plotting.plot_intents[{i}] is too vague ('{intent_s}'). "
                                "Use concrete field-based intents only."
                            )

                    referenced_fields = []
                    for field_name in sorted(known_plot_fields, key=len, reverse=True):
                        if field_name.lower() in intent_l:
                            referenced_fields.append(field_name)

                    if len(referenced_fields) == 0:
                        raise ValueError(
                            f"plan.plotting.plot_intents[{i}] must reference at least one concrete retrieved or derived field. "
                            f"Got '{intent_s}'."
                        )

                    invalid_references = [
                        field_name
                        for field_name in referenced_fields
                        if field_name not in allowed_direct_plot_fields and field_name not in allowed_derived_plot_fields
                    ]
                    if invalid_references:
                        raise ValueError(
                            f"plan.plotting.plot_intents[{i}] contains unsupported fields: {invalid_references}"
                        )

                    words = [
                        w.strip(" ,.:;()[]{}").lower()
                        for w in intent_s.split()
                        if w.strip(" ,.:;()[]{}")
                    ]
                    unknown_words = []
                    for w in words:
                        if w in known_plot_words:
                            continue
                        if w in {f.lower() for f in known_plot_fields}:
                            continue
                        unknown_words.append(w)

                    if unknown_words:
                        pass

                    for field_name in referenced_fields:
                        if field_name in allowed_derived_plot_fields:
                            continue
                        if field_name not in retrieved_fields:
                            raise ValueError(
                                f"plan.plotting.plot_intents[{i}] references field '{field_name}' "
                                "that is not present in plan.retrieval.fields."
                            )

                got_ids = {s.get("id") for s in steps if isinstance(s, dict)}
                required_plot_steps = [
                    "plot_suggest",
                    "plot_select",
                    "plot_generate_v1",
                    "plot_validate",
                    "plot_generate_v2",
                ]
                missing_plot = [x for x in required_plot_steps if x not in got_ids]
                if missing_plot:
                    raise ValueError(f"Missing plotting step(s): {missing_plot}")

        documentation = plan.get("documentation", None)
        if documentation is not None:
            if not isinstance(documentation, dict):
                raise ValueError("plan.documentation must be a dict.")
            if not isinstance(documentation.get("enabled"), bool):
                raise ValueError("plan.documentation.enabled must be a bool.")
            if documentation["enabled"]:
                if not isinstance(documentation.get("template"), str) or not documentation["template"].strip():
                    raise ValueError("plan.documentation.template must be a non-empty string.")
                if documentation.get("output") != "PDF":
                    raise ValueError('plan.documentation.output must be "PDF".')

                got_ids = {s.get("id") for s in steps if isinstance(s, dict)}
                required_doc_steps = ["latex_write", "latex_compile"]
                missing_doc = [x for x in required_doc_steps if x not in got_ids]
                if missing_doc:
                    raise ValueError(f"Missing documentation step(s): {missing_doc}")

        for s in steps:
            if "agent" in s and not (isinstance(s["agent"], str) and s["agent"].strip()):
                raise ValueError(f"Step '{s.get('id')}' has invalid 'agent' (must be non-empty string).")
            if "inputs" in s and not (
                isinstance(s["inputs"], list) and all(isinstance(x, str) and x.strip() for x in s["inputs"])
            ):
                raise ValueError(f"Step '{s.get('id')}' has invalid 'inputs' (must be list[str]).")
            if "outputs" in s and not (
                isinstance(s["outputs"], list) and all(isinstance(x, str) and x.strip() for x in s["outputs"])
            ):
                raise ValueError(f"Step '{s.get('id')}' has invalid 'outputs' (must be list[str]).")
            if "constraints" in s and not isinstance(s["constraints"], dict):
                raise ValueError(f"Step '{s.get('id')}' has invalid 'constraints' (must be dict).")

    except ValueError as e:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in planner_tool: invalid plan format: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    context_variables["plan"] = plan
    context_variables["plan_description"] = plan_description.strip()
    context_variables["success_criteria"] = [x.strip() for x in success_criteria]
    context_variables["plan_score"] = plan_score_f

    next_agent = plan["routing"]["after_planner"]
    context_variables["next_agent"] = next_agent

    message = {
        "plan_description": context_variables["plan_description"],
        "success_criteria": context_variables["success_criteria"],
        "plan_score": context_variables["plan_score"],
        "plan": plan,
    }

    return ReplyResult(
        message=json.dumps(message, indent=2, ensure_ascii=False, default=str),
        target=AgentNameTarget(next_agent),
        context_variables=context_variables,
    )

# Tool Reviewer — review_plan

def review_plan_tool(
    decision: Annotated[
        Literal["approve", "revise", "handoff"],
        "approve -> go to retriever; revise -> go back to planner; handoff -> go back to human.",
    ],
    review_notes: Annotated[str, "Short, concrete notes about what is wrong/right in the plan."],
    context_variables: ContextVariables,
) -> ReplyResult:
    agent_name = "AgentPlanReviewer"
    plan = context_variables.get("plan")

    fallback_next = "AgentPlanner"
    if isinstance(plan, dict):
        routing_safe = plan.get("routing")
        if isinstance(routing_safe, dict):
            fallback_next = routing_safe.get("on_revise", fallback_next)

    if context_variables.get("plan_review_status") == "completed":
        existing_review_notes = context_variables.get("review_notes", "")
        existing_review_status = context_variables.get("review_status", None)

        next_agent_existing = fallback_next
        if isinstance(plan, dict):
            routing_existing = plan.get("routing")
            if isinstance(routing_existing, dict):
                if existing_review_status == "approved":
                    next_agent_existing = routing_existing.get("on_approve", next_agent_existing)
                elif existing_review_status == "handoff":
                    next_agent_existing = routing_existing.get("on_handoff", next_agent_existing)
                else:
                    next_agent_existing = routing_existing.get("on_revise", next_agent_existing)

        context_variables["next_agent"] = next_agent_existing

        if existing_review_status == "approved":
            message_prefix = "Plan review -> approved"
        elif existing_review_status == "handoff":
            message_prefix = "Plan review -> handoff"
        else:
            message_prefix = "Plan review -> revise"

        return ReplyResult(
            message=f"{message_prefix}\n{existing_review_notes}",
            target=AgentNameTarget(next_agent_existing),
            context_variables=context_variables,
        )

    context_variables["plan_review_status"] = "running"

    try:
        if plan is None or not isinstance(plan, dict):
            raise ValueError("Missing or invalid plan in context_variables['plan'].")
        if decision not in {"approve", "revise", "handoff"}:
            raise ValueError("decision must be one of: approve, revise, handoff.")
        if not isinstance(review_notes, str) or not review_notes.strip():
            raise ValueError("review_notes must be a non-empty string.")
    except ValueError as e:
        context_variables["plan_review_status"] = "failed"
        context_variables["review_status"] = "revise"
        context_variables["review_notes"] = f"{str(e)}"
        context_variables["next_agent"] = fallback_next
        return ReplyResult(
            message=f"Plan review -> revise\n{context_variables['review_notes']}",
            target=AgentNameTarget(fallback_next),
            context_variables=context_variables,
        )

    try:
        if not isinstance(plan.get("schema_version"), str) or not plan["schema_version"].strip():
            raise ValueError("plan.schema_version must be a non-empty string.")
        if not isinstance(plan.get("main_task_improved"), str) or not plan["main_task_improved"].strip():
            raise ValueError("plan.main_task_improved must be a non-empty string.")

        routing = plan.get("routing")
        if not isinstance(routing, dict):
            raise ValueError("plan.routing must be a dict.")
        for k in ("after_planner", "on_approve", "on_revise", "on_handoff"):
            if not isinstance(routing.get(k), str) or not routing[k].strip():
                raise ValueError(f"plan.routing.{k} must be a non-empty string.")

        steps = plan.get("steps")
        if not isinstance(steps, list) or len(steps) == 0:
            raise ValueError("plan.steps must be a non-empty list.")

        steps_by_id = {}
        for s in steps:
            if not isinstance(s, dict):
                raise ValueError("Each step in plan.steps must be a dict.")
            sid = s.get("id")
            if not isinstance(sid, str) or not sid.strip():
                raise ValueError("Each step must have a non-empty 'id'.")
            if not isinstance(s.get("agent"), str) or not s["agent"].strip():
                raise ValueError(f"Step '{sid}' must have a non-empty 'agent'.")
            steps_by_id[sid] = s

        guardrails = plan.get("guardrails", None)
        if guardrails is not None and not isinstance(guardrails, dict):
            raise ValueError("plan.guardrails must be a dict if present.")

        required_ids = []
        if isinstance(guardrails, dict):
            required_ids = guardrails.get("required_step_ids", [])

        if required_ids:
            if not isinstance(required_ids, list) or not all(isinstance(x, str) and x.strip() for x in required_ids):
                raise ValueError("plan.guardrails.required_step_ids must be a list of non-empty strings.")
            missing = sorted(list(set(required_ids) - set(steps_by_id.keys())))
            if missing:
                raise ValueError(f"Missing required step(s): {missing}")

        retrieval = plan.get("retrieval")
        if not isinstance(retrieval, dict):
            raise ValueError("plan.retrieval must be a dict.")
        if not isinstance(retrieval.get("search_criteria"), dict):
            raise ValueError("plan.retrieval.search_criteria must be a dict.")
        fields = retrieval.get("fields")
        if not isinstance(fields, list) or not fields or not all(isinstance(x, str) and x.strip() for x in fields):
            raise ValueError("plan.retrieval.fields must be a non-empty list of non-empty strings.")
        sn = retrieval.get("sample_number")
        if not isinstance(sn, int) or sn <= 0:
            raise ValueError("plan.retrieval.sample_number must be a positive int.")

        plotting = plan.get("plotting", None)
        if plotting is not None and not isinstance(plotting, dict):
            raise ValueError("plan.plotting must be a dict if present.")
        if isinstance(plotting, dict):
            enabled = plotting.get("enabled", None)
            if not isinstance(enabled, bool):
                raise ValueError("plan.plotting.enabled must be a bool.")
            if enabled is True:
                n_candidates = plotting.get("n_candidates", None)
                n_selected = plotting.get("n_selected", None)
                output_dir = plotting.get("output_dir", None)

                if not isinstance(n_candidates, int) or n_candidates != 10:
                    raise ValueError("plan.plotting.n_candidates must be int == 10.")
                if not isinstance(n_selected, int) or n_selected != 5:
                    raise ValueError("plan.plotting.n_selected must be int == 5.")
                if not isinstance(output_dir, str) or output_dir.strip() != "plots":
                    raise ValueError('plan.plotting.output_dir must be "plots".')

                required_plot_steps = [
                    "plot_suggest",
                    "plot_select",
                    "plot_generate_v1",
                    "plot_validate",
                    "plot_generate_v2",
                ]
                missing_plot = [x for x in required_plot_steps if x not in steps_by_id]
                if missing_plot:
                    raise ValueError(f"Missing plotting step(s): {missing_plot}")

        defaults = plan.get("defaults", {})
        if defaults and not isinstance(defaults, dict):
            raise ValueError("plan.defaults must be a dict if present.")

        if isinstance(guardrails, dict):
            sn_guard = guardrails.get("sample_number")
            if sn_guard is not None:
                if not isinstance(sn_guard, dict):
                    raise ValueError("plan.guardrails.sample_number must be a dict.")
                for kk in ("default", "min", "max"):
                    if kk not in sn_guard:
                        raise ValueError(f"plan.guardrails.sample_number missing key '{kk}'.")
                    if not isinstance(sn_guard[kk], int):
                        raise ValueError(f"plan.guardrails.sample_number.{kk} must be an int.")
                if sn_guard["min"] <= 0:
                    raise ValueError("plan.guardrails.sample_number.min must be > 0.")
                if sn_guard["max"] < sn_guard["min"]:
                    raise ValueError("plan.guardrails.sample_number.max must be >= min.")
                if not (sn_guard["min"] <= sn_guard["default"] <= sn_guard["max"]):
                    raise ValueError("plan.guardrails.sample_number.default must be within [min, max].")

    except ValueError as e:
        context_variables["plan_review_status"] = "failed"
        context_variables["review_status"] = "revise"
        context_variables["review_notes"] = f"{review_notes.strip()} | validation_error: {e}"
        next_agent = plan["routing"]["on_revise"] if isinstance(plan.get("routing"), dict) else "AgentPlanner"
        context_variables["next_agent"] = next_agent
        return ReplyResult(
            message=f"Plan review -> revise\n{context_variables['review_notes']}",
            target=AgentNameTarget(next_agent),
            context_variables=context_variables,
        )

    context_variables["review_notes"] = review_notes.strip()

    if decision == "approve":
        next_agent = plan["routing"]["on_approve"]
        context_variables["review_status"] = "approved"
        context_variables["plan_review_status"] = "completed"
        context_variables["next_agent"] = next_agent
        return ReplyResult(
            message=f"Plan review -> approved\n{context_variables['review_notes']}",
            target=AgentNameTarget(next_agent),
            context_variables=context_variables,
        )

    if decision == "revise":
        next_agent = plan["routing"]["on_revise"]
        context_variables["review_status"] = "revise"
        context_variables["plan_review_status"] = "completed"
        context_variables["next_agent"] = next_agent
        return ReplyResult(
            message=f"Plan review -> revise\n{context_variables['review_notes']}",
            target=AgentNameTarget(next_agent),
            context_variables=context_variables,
        )

    next_agent = plan["routing"]["on_handoff"]
    context_variables["review_status"] = "handoff"
    context_variables["plan_review_status"] = "completed"
    context_variables["next_agent"] = next_agent
    return ReplyResult(
        message=f"Plan review -> handoff\n{context_variables['review_notes']}",
        target=AgentNameTarget(next_agent),
        context_variables=context_variables,
    )

# Tool — materials_project_retriever

def material_retriever(
    context_variables: ContextVariables,
    search_criteria: Optional[dict] = None,
    fields: Optional[List[str]] = None,
    sample_number: Optional[int] = None,
    plan: Optional[dict] = None,
) -> ReplyResult:
    """Retrieve materials from Materials Project using plan.retrieval.search_criteria / fields / sample_number."""

    agent_name = "AgentMaterialsRetriever"

    if plan is not None:
        if isinstance(plan, dict) and plan:
            context_variables["plan"] = plan
        else:
            context_variables["next_agent"] = "AgentPlanner"
            return ReplyResult(
                message="Error in material_retriever: plan must be a non-empty dict if provided.",
                target=AgentNameTarget("AgentPlanner"),
                context_variables=context_variables,
            )

    plan = context_variables.get("plan", {})
    if not isinstance(plan, dict):
        plan = {}

    routing = plan.get("routing", {})
    if not isinstance(routing, dict):
        routing = {}
    next_on_revise = routing.get("on_revise", "AgentPlanner")

    retrieval = plan.get("retrieval", {})
    if not isinstance(retrieval, dict):
        retrieval = {}

    try:
        if not isinstance(plan, dict) or not plan:
            raise ValueError("Missing or invalid plan in context_variables['plan'].")
        if not isinstance(retrieval, dict) or not retrieval:
            raise ValueError("Missing plan['retrieval'] dict.")

        search_criteria = retrieval.get("search_criteria", None)
        fields = retrieval.get("fields", None)
        sample_number = retrieval.get("sample_number", None)

        if not isinstance(search_criteria, dict):
            raise ValueError("plan['retrieval']['search_criteria'] must be a dict.")
        if not isinstance(fields, list) or len(fields) == 0 or not all(isinstance(f, str) and f.strip() for f in fields):
            raise ValueError("plan['retrieval']['fields'] must be a non-empty list of non-empty strings.")
        if not isinstance(sample_number, int) or sample_number <= 0:
            raise ValueError("plan['retrieval']['sample_number'] must be a positive int.")
    except ValueError as e:
        context_variables["mp_results"] = []
        context_variables["search_criteria"] = {}
        context_variables["fields"] = []
        context_variables["sample_number"] = 0
        context_variables["next_agent"] = next_on_revise
        return ReplyResult(
            message=f"Error in material_retriever: {e}",
            target=AgentNameTarget(next_on_revise),
            context_variables=context_variables,
        )

    guardrails = plan.get("guardrails", {}) if isinstance(plan, dict) else {}
    sn_guard = guardrails.get("sample_number") if isinstance(guardrails, dict) else None

    if isinstance(sn_guard, dict):
        sn_min = sn_guard.get("min")
        sn_max = sn_guard.get("max")
        if isinstance(sn_min, int) and sample_number < sn_min:
            sample_number = sn_min
        if isinstance(sn_max, int) and sample_number > sn_max:
            sample_number = sn_max

    allowed_search_keys = None
    allowed_fields = None

    if isinstance(guardrails, dict):
        ak = guardrails.get("allowed_search_keys")
        af = guardrails.get("allowed_fields")
        if isinstance(ak, list) and all(isinstance(x, str) and x.strip() for x in ak):
            allowed_search_keys = set(x.strip() for x in ak)
        if isinstance(af, list) and all(isinstance(x, str) and x.strip() for x in af):
            allowed_fields = set(x.strip() for x in af)

    if allowed_search_keys is None:
        allowed_search_keys = {
            "band_gap",
            "energy_above_hull",
            "density",
            "num_sites",
            "k_voigt",
            "g_voigt",
            "elements",
            "chemsys",
            "exclude_elements",
        }

    if allowed_fields is None:
        allowed_fields = {
            "material_id",
            "formula_pretty",
            "band_gap",
            "energy_above_hull",
            "density",
            "volume",
            "symmetry",
            "nsites",
            "elements",
            "chemsys",
            "is_stable",
            "bulk_modulus",
            "shear_modulus",
        }

    try:
        if not all(isinstance(k, str) and k.strip() for k in search_criteria.keys()):
            raise ValueError("search_criteria keys must be non-empty strings.")

        if "excluded_elements" in search_criteria:
            raise ValueError(
                "Use 'exclude_elements' instead of 'excluded_elements' in plan['retrieval']['search_criteria']."
            )

        bad_keys = [k for k in search_criteria.keys() if k not in allowed_search_keys]
        if bad_keys:
            raise ValueError(
                f"Unsupported search_criteria key(s): {bad_keys}. "
                f"Allowed keys: {sorted(list(allowed_search_keys))}"
            )

        bad_fields = [f for f in fields if f not in allowed_fields]
        if bad_fields:
            raise ValueError(
                f"Unsupported field(s): {bad_fields}. "
                f"Allowed fields: {sorted(list(allowed_fields))}"
            )

    except ValueError as e:
        context_variables["mp_results"] = []
        context_variables["search_criteria"] = search_criteria
        context_variables["fields"] = fields
        context_variables["sample_number"] = 0
        context_variables["next_agent"] = next_on_revise
        return ReplyResult(
            message=f"Error in material_retriever: {e}",
            target=AgentNameTarget(next_on_revise),
            context_variables=context_variables,
        )

    alias_map = {
        "num_sites": "nsites",
        "k_voigt": "bulk_modulus",
        "g_voigt": "shear_modulus",
        "pretty_formula": "formula_pretty",
    }

    def _is_range_tuple(v) -> bool:
        return isinstance(v, tuple) and len(v) == 2

    def _normalize_numeric_range(v):
        if _is_range_tuple(v):
            lo, hi = v
            if lo is None or hi is None:
                raise ValueError("Numeric range filters must be (min, max) with both values set.")
            if not isinstance(lo, (int, float)) or not isinstance(hi, (int, float)):
                raise ValueError("Numeric range filters must be (min, max) numbers.")
            if lo > hi:
                raise ValueError("Numeric range filters must satisfy min <= max.")
            return (lo, hi)

        if isinstance(v, (int, float)):
            return (v, v)

        return v

    normalized_criteria = {}
    try:
        for key, value in search_criteria.items():
            mapped_key = alias_map.get(key, key)

            if mapped_key in {"band_gap", "energy_above_hull", "density", "nsites", "bulk_modulus", "shear_modulus"}:
                normalized_criteria[mapped_key] = _normalize_numeric_range(value)
                continue

            if mapped_key in {"elements", "chemsys", "exclude_elements"}:
                if not (
                    isinstance(value, list)
                    and len(value) > 0
                    and all(isinstance(x, str) and x.strip() for x in value)
                ):
                    raise ValueError(f"{key} must be a non-empty list of non-empty strings.")
                normalized_criteria[mapped_key] = [x.strip() for x in value]
                continue

            normalized_criteria[mapped_key] = value
    except ValueError as e:
        context_variables["mp_results"] = []
        context_variables["search_criteria"] = search_criteria
        context_variables["fields"] = fields
        context_variables["sample_number"] = 0
        context_variables["next_agent"] = next_on_revise
        return ReplyResult(
            message=f"Error in material_retriever: {e}",
            target=AgentNameTarget(next_on_revise),
            context_variables=context_variables,
        )

    api_key = os.getenv("MP_API_KEY")
    if not api_key:
        context_variables["mp_results"] = []
        context_variables["search_criteria"] = search_criteria
        context_variables["fields"] = fields
        context_variables["sample_number"] = 0
        context_variables["next_agent"] = next_on_revise
        return ReplyResult(
            message="Error in material_retriever: Missing MP_API_KEY environment variable.",
            target=AgentNameTarget(next_on_revise),
            context_variables=context_variables,
        )

    try:
        with MPRester(api_key) as mpr:
            all_results = list(
                mpr.materials.summary.search(
                    fields=fields,
                    **normalized_criteria,
                )
            )
    except Exception as e:
        context_variables["mp_results"] = []
        context_variables["search_criteria"] = search_criteria
        context_variables["fields"] = fields
        context_variables["sample_number"] = 0
        context_variables["next_agent"] = next_on_revise
        return ReplyResult(
            message=f"Error in material_retriever while querying Materials Project: {e}",
            target=AgentNameTarget(next_on_revise),
            context_variables=context_variables,
        )

    if not all_results:
        context_variables["mp_results"] = []
        context_variables["search_criteria"] = search_criteria
        context_variables["fields"] = fields
        context_variables["sample_number"] = 0
        context_variables["next_agent"] = next_on_revise
        return ReplyResult(
            message="No materials found for the given search_criteria. Revise filters while keeping the main intent.",
            target=AgentNameTarget(next_on_revise),
            context_variables=context_variables,
        )

    results = all_results[:sample_number]

    next_agent = "AgentAnalyzer"
    steps = plan.get("steps")
    if isinstance(steps, list):
        for s in steps:
            if isinstance(s, dict) and s.get("id") == "analyze":
                a = s.get("agent")
                if isinstance(a, str) and a.strip():
                    next_agent = a.strip()
                break

    context_variables["search_criteria"] = search_criteria
    context_variables["fields"] = fields
    context_variables["sample_number"] = len(results)
    context_variables["mp_results"] = results
    context_variables["next_agent"] = next_agent

    return ReplyResult(
        message=f"Retrieved {len(results)} materials from Materials Project with search_criteria={search_criteria}.",
        target=AgentNameTarget(next_agent),
        context_variables=context_variables,
    )

# Tool — final_conclusion_tool (Analyzer)

def final_conclusion_tool(
    context_variables: ContextVariables,
    final_conclusion: str | None = None,
) -> ReplyResult:
    agent_name = "AgentAnalyzer"
    results = context_variables.get("mp_results", None)

    plan = context_variables.get("plan", {})
    routing = plan.get("routing", {}) if isinstance(plan, dict) else {}
    next_on_revise = routing.get("on_revise", "AgentPlanner")

    if results is None or results == []:
        context_variables["next_agent"] = next_on_revise
        return ReplyResult(
            message='{"error":"No materials available in context_variables[\\"mp_results\\"]."}',
            target=AgentNameTarget(next_on_revise),
            context_variables=context_variables,
        )

    if not isinstance(results, list):
        context_variables["next_agent"] = next_on_revise
        return ReplyResult(
            message='{"error":"context_variables[\\"mp_results\\"] must be a list."}',
            target=AgentNameTarget(next_on_revise),
            context_variables=context_variables,
        )

    results = [r for r in results if r is not None]
    if len(results) == 0:
        context_variables["next_agent"] = next_on_revise
        return ReplyResult(
            message='{"error":"context_variables[\\"mp_results\\"] contains no valid entries."}',
            target=AgentNameTarget(next_on_revise),
            context_variables=context_variables,
        )

    main_task = context_variables.get("main_task_improved", "")
    if not isinstance(main_task, str) or not main_task.strip():
        main_task = ""

    def _to_float(x):
        try:
            if x is None:
                return None
            return float(x)
        except Exception:
            return None

    def get_value(entry, k):
        if isinstance(entry, dict):
            return entry.get(k)
        return getattr(entry, k, None)

    def _percentiles(vals, ps):
        if not vals:
            return {p: None for p in ps}
        s = sorted(vals)
        n = len(s)
        out = {}
        for p in ps:
            if n == 1:
                out[p] = s[0]
                continue
            q = (p / 100.0) * (n - 1)
            lo = int(q)
            hi = min(lo + 1, n - 1)
            frac = q - lo
            out[p] = s[lo] * (1 - frac) + s[hi] * frac
        return out

    def _jsonable(x):
        if x is None:
            return None
        if isinstance(x, (str, int, float, bool)):
            return x
        if isinstance(x, (list, tuple)):
            return [_jsonable(v) for v in x]
        if isinstance(x, dict):
            out = {}
            for k, v in x.items():
                out[str(k)] = _jsonable(v)
            return out

        if hasattr(x, "model_dump"):
            try:
                d = x.model_dump()
                return _jsonable(d)
            except Exception:
                pass

        if hasattr(x, "as_dict"):
            try:
                d = x.as_dict()
                return _jsonable(d)
            except Exception:
                pass

        if hasattr(x, "__dict__"):
            try:
                return _jsonable(vars(x))
            except Exception:
                pass

        try:
            return str(x)
        except Exception:
            return None

    def _as_material_dict(x):
        if isinstance(x, dict):
            d = dict(x)
        elif hasattr(x, "model_dump"):
            try:
                d = x.model_dump()
            except Exception:
                d = {}
        elif hasattr(x, "dict"):
            try:
                d = x.dict()
            except Exception:
                d = {}
        else:
            keys = [
                "material_id",
                "formula_pretty",
                "band_gap",
                "energy_above_hull",
                "e_above_hull",
                "density",
                "elements",
                "is_stable",
            ]
            d = {}
            for k in keys:
                try:
                    d[k] = getattr(x, k, None)
                except Exception:
                    d[k] = None

        return _jsonable(d)

    def _build_analyzer_error(message: str) -> ReplyResult:
        forward = context_variables.get("next_agent", "AgentPlotSuggester") or "AgentPlotSuggester"
        return ReplyResult(
            message=json.dumps({"error": message}, ensure_ascii=False),
            target=AgentNameTarget(forward),
            context_variables=context_variables,
        )

    def _contains_any(text: str, patterns: list[str]) -> str | None:
        lowered = text.lower()
        for pattern in patterns:
            if pattern in lowered:
                return pattern
        return None

    def _validate_final_conclusion(
        text: str,
        *,
        n_total: int,
        bg_missing: int,
        eah_missing: int,
        dens_missing: int,
        stable_count: int,
        near_stable_count: int,
        meta_count: int,
        unstable_count: int,
        bg_bins: dict,
    ) -> str | None:
        lowered = text.lower()

        forbidden_global_claims = [
            "all materials meet",
            "all retrieved materials meet",
            "all candidates meet",
            "materials meet the criteria",
            "retrieved materials meet the criteria",
            "candidates meet the criteria",
            "materials satisfy the criteria",
            "retrieved materials satisfy the criteria",
            "candidates satisfy the criteria",
            "materials are suitable",
            "retrieved materials are suitable",
            "candidates are suitable",
            "the retrieved materials are suitable",
            "the materials are suitable",
            "all materials are suitable",
            "all retrieved materials are suitable",
        ]

        matched = _contains_any(lowered, forbidden_global_claims)
        if matched is not None:
            return (
                f"final_conclusion contains an unsupported suitability/compliance claim ('{matched}'). "
                "Use only data-grounded wording."
            )

        if eah_missing > 0:
            forbidden_missing_eah_claims = [
                "with energy_above_hull, band_gap, and density within specified ranges",
                "with energy_above_hull within specified ranges",
                "with energy_above_hull within the specified range",
                "all retrieved materials have energy_above_hull",
                "all materials have energy_above_hull",
                "all 100 materials",
                f"all {n_total} materials",
                f"retrieved {n_total} materials with energy_above_hull",
            ]
            matched = _contains_any(lowered, forbidden_missing_eah_claims)
            if matched is not None:
                return (
                    "final_conclusion overstates field completeness despite missing energy_above_hull values "
                    f"({eah_missing} missing). Problematic phrase: '{matched}'."
                )

        if bg_missing > 0:
            forbidden_missing_bg_claims = [
                "all retrieved materials have band_gap",
                "all materials have band_gap",
            ]
            matched = _contains_any(lowered, forbidden_missing_bg_claims)
            if matched is not None:
                return (
                    "final_conclusion overstates field completeness despite missing band_gap values "
                    f"({bg_missing} missing). Problematic phrase: '{matched}'."
                )

        if dens_missing > 0:
            forbidden_missing_density_claims = [
                "all retrieved materials have density",
                "all materials have density",
            ]
            matched = _contains_any(lowered, forbidden_missing_density_claims)
            if matched is not None:
                return (
                    "final_conclusion overstates field completeness despite missing density values "
                    f"({dens_missing} missing). Problematic phrase: '{matched}'."
                )

        explicitly_claims_all_near_stable = _contains_any(
            lowered,
            [
                "all are near-stable",
                "all materials are near-stable",
                "all retrieved materials are near-stable",
                "all candidates are near-stable",
                "all are stable or near-stable",
                "all materials are stable or near-stable",
            ],
        )
        if explicitly_claims_all_near_stable is not None:
            allowed_total = stable_count + near_stable_count
            if allowed_total != n_total or eah_missing > 0 or meta_count > 0 or unstable_count > 0:
                return (
                    f"final_conclusion claims universal stability/near-stability ('{explicitly_claims_all_near_stable}') "
                    "but the structured summary does not support that."
                )

        very_wide = int(bg_bins.get("very_wide_bandgap", 0))
        wide = int(bg_bins.get("wide_bandgap", 0))
        semiconductors = int(bg_bins.get("semiconductor", 0))
        narrow = int(bg_bins.get("narrow_gap", 0))
        gapless = int(bg_bins.get("gapless_or_metal_like", 0))
        bg_available = wide + very_wide + semiconductors + narrow + gapless

        if "predominantly wide" in lowered or "mostly wide" in lowered:
            if bg_available == 0 or (wide + very_wide) <= (bg_available / 2):
                return (
                    "final_conclusion claims band gaps are predominantly wide, "
                    "but that is not supported by band_gap_bins."
                )

        return None

    serializable_results = [_as_material_dict(r) for r in results]

    bg_vals = []
    eah_vals = []
    dens_vals = []

    bg_missing = 0
    eah_missing = 0
    dens_missing = 0

    for entry in serializable_results:
        bg = _to_float(get_value(entry, "band_gap"))
        eah_raw = get_value(entry, "energy_above_hull")
        if eah_raw is None:
            eah_raw = get_value(entry, "e_above_hull")
        eah = _to_float(eah_raw)
        dens = _to_float(get_value(entry, "density"))

        bg_vals.append(bg)
        eah_vals.append(eah)
        dens_vals.append(dens)

        if bg is None:
            bg_missing += 1
        if eah is None:
            eah_missing += 1
        if dens is None:
            dens_missing += 1

    bg_clean = [v for v in bg_vals if isinstance(v, (int, float))]
    eah_clean = [v for v in eah_vals if isinstance(v, (int, float))]
    dens_clean = [v for v in dens_vals if isinstance(v, (int, float))]

    n_total = len(serializable_results)

    stable_count = 0
    near_stable_count = 0
    meta_count = 0
    unstable_count = 0

    for v in eah_clean:
        if v <= 1e-6:
            stable_count += 1
        elif v <= 0.05:
            near_stable_count += 1
        elif v <= 0.2:
            meta_count += 1
        else:
            unstable_count += 1

    bg_bins = {
        "gapless_or_metal_like": 0,
        "narrow_gap": 0,
        "semiconductor": 0,
        "wide_bandgap": 0,
        "very_wide_bandgap": 0,
    }

    for v in bg_clean:
        if v < 0.1:
            bg_bins["gapless_or_metal_like"] += 1
        elif v < 1.0:
            bg_bins["narrow_gap"] += 1
        elif v < 3.0:
            bg_bins["semiconductor"] += 1
        elif v <= 6.0:
            bg_bins["wide_bandgap"] += 1
        else:
            bg_bins["very_wide_bandgap"] += 1

    bg_p = _percentiles(bg_clean, [10, 50, 90])
    eah_p = _percentiles(eah_clean, [10, 50, 90])
    dens_p = _percentiles(dens_clean, [10, 50, 90])

    output = {
        "schema_version": "1.1",
        "created_at": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
        "main_task_improved": main_task.strip(),
        "counts": {
            "materials_retrieved": n_total,
            "missing": {
                "band_gap": bg_missing,
                "energy_above_hull": eah_missing,
                "density": dens_missing,
            },
        },
        "ranges": {
            "band_gap_eV": {
                "p10": bg_p[10],
                "p50": bg_p[50],
                "p90": bg_p[90],
            },
            "energy_above_hull_eV_per_atom": {
                "p10": eah_p[10],
                "p50": eah_p[50],
                "p90": eah_p[90],
            },
            "density_g_cm3": {
                "p10": dens_p[10],
                "p50": dens_p[50],
                "p90": dens_p[90],
            },
        },
        "patterns": {
            "stability_bins": {
                "stable_eah_eq_0": stable_count,
                "near_stable_le_0p05": near_stable_count,
                "metastable_0p05_to_0p2": meta_count,
                "likely_unstable_gt_0p2": unstable_count,
            },
            "band_gap_bins": bg_bins,
        },
        "limitations": [
            "Summary uses only retrieved fields (band_gap, energy_above_hull, density, volume if present).",
            "No thermal transport, melting point, oxidation kinetics, toxicity, or mechanical validation included.",
        ],
    }

    if not isinstance(final_conclusion, str) or not final_conclusion.strip():
        return _build_analyzer_error(
            "final_conclusion must be a non-empty string generated by AgentAnalyzer."
        )

    concise = final_conclusion.strip()

    validation_error = _validate_final_conclusion(
        concise,
        n_total=n_total,
        bg_missing=bg_missing,
        eah_missing=eah_missing,
        dens_missing=dens_missing,
        stable_count=stable_count,
        near_stable_count=near_stable_count,
        meta_count=meta_count,
        unstable_count=unstable_count,
        bg_bins=bg_bins,
    )

    if validation_error is not None:
        return _build_analyzer_error(
            f"final_conclusion is not consistent with analysis_summary: {validation_error}"
        )

    context_variables["analysis_summary"] = output
    context_variables["final_conclusion"] = concise

    plotting_enabled = False
    if isinstance(plan, dict):
        plotting = plan.get("plotting", {})
        if isinstance(plotting, dict):
            plotting_enabled = bool(plotting.get("enabled", False))

    next_agent = "AgentCoder"
    if plotting_enabled:
        steps = plan.get("steps", [])
        steps_by_id = {}
        if isinstance(steps, list):
            for s in steps:
                if isinstance(s, dict) and isinstance(s.get("id"), str) and s["id"].strip():
                    steps_by_id[s["id"].strip()] = s

        plot_suggest_step = steps_by_id.get("plot_suggest", {})
        candidate_agent = plot_suggest_step.get("agent", None)

        if isinstance(candidate_agent, str) and candidate_agent.strip():
            next_agent = candidate_agent.strip()
        else:
            next_agent = "AgentPlotSuggester"

    context_variables["next_agent"] = next_agent

    project_folder = os.environ.get("PROJECT_FOLDER", "ag2_project")
    os.makedirs(project_folder, exist_ok=True)

    final_path = os.path.join(project_folder, "final_conclusion.json")
    with open(final_path, "w", encoding="utf-8") as f:
        json.dump(
            {"analysis_summary": output, "final_conclusion": concise},
            f,
            indent=2,
            ensure_ascii=False,
        )

    materials_path = os.path.join(project_folder, "materials_data.json")
    payload = {
        "mp_results": serializable_results,
        "analysis_summary": output,
        "final_conclusion": concise,
    }
    with open(materials_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

    context_variables["materials_data_path"] = materials_path

    return ReplyResult(
        message=json.dumps(
            {"analysis_summary": output, "final_conclusion": concise},
            indent=2,
            ensure_ascii=False,
        ),
        target=AgentNameTarget(next_agent),
        context_variables=context_variables,
    )
    
# Tool — plot_suggester_tool

def plot_suggester_tool(
    candidates: Annotated[List[dict], "List of plot candidate dicts (must be length 10)."],
    context_variables: ContextVariables,
) -> ReplyResult:
    agent_name = "AgentPlotSuggester"

    _MAX_SUGGESTER_RETRIES = 2
    _suggester_retries = int(context_variables.get("suggester_retries", 0))

    plan = context_variables.get("plan", {})
    routing = plan.get("routing", {}) if isinstance(plan, dict) else {}
    next_agent = routing.get("after_plot_suggest", None)
    if not isinstance(next_agent, str) or not next_agent.strip():
        next_agent = "AgentPlotSelector"

    def _error(message: str) -> ReplyResult:
        nonlocal _suggester_retries
        _suggester_retries += 1
        context_variables["suggester_retries"] = _suggester_retries
        context_variables["next_agent"] = agent_name
        context_variables["plot_candidates_error"] = message
        context_variables["plot_suggest_status"] = "failed"
        if _suggester_retries > _MAX_SUGGESTER_RETRIES:
            return ReplyResult(
                message=(
                    f"Error in plot_suggester_tool: AgentPlotSuggester exceeded {_MAX_SUGGESTER_RETRIES} retries. "
                    "Handing off to Human for manual intervention."
                ),
                target=AgentNameTarget("Human"),
                context_variables=context_variables,
            )
        return ReplyResult(
            message=f"Error in plot_suggester_tool: {message}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    try:
        if not isinstance(candidates, list):
            return _error(
                "candidates must be a list of 10 items. "
                "Rebuild and resubmit one complete valid set of exactly 10 candidates; do not patch a single candidate in isolation."
            )

        if len(candidates) != 10:
            return _error(
                f"candidates must be a list of 10 items. Got {len(candidates)}. "
                "If you remove one candidate, replace it in the same response. Rebuild the full 10-candidate set before retrying."
            )

        retrieval = plan.get("retrieval", {}) if isinstance(plan, dict) else {}
        retrieval_fields = retrieval.get("fields", []) if isinstance(retrieval, dict) else []
        if not isinstance(retrieval_fields, list):
            retrieval_fields = []

        available_direct_fields = {x.strip() for x in retrieval_fields if isinstance(x, str) and x.strip()}
        allowed_direct_fields = {"band_gap", "density", "energy_above_hull"}
        available_direct_fields = {x for x in available_direct_fields if x in allowed_direct_fields}

        allowed_derived_fields = set()
        allowed_axis_count_labels = set()

        forbidden_derived_aliases = {
            "stability_bin",
            "band_gap_bin",
            "stability_bucket",
            "band_gap_bucket",
            "counts",
            "material_count",
            "number_of_materials",
            "materials_count",
            "stability_bins",
            "band_gap_bins",
            "count",
            "frequency",
        }

        required_keys = ["plot_id", "title", "description", "data_requirements", "axes", "rationale", "plot_type"]

        seen_ids = set()
        normalized = []

        expected_ids = [f"plot_{i}" for i in range(1, 11)]

        derived_plot_count = 0
        derived_bar_count = 0
        direct_plot_count = 0

        for i, item in enumerate(candidates):
            if not isinstance(item, dict):
                return _error(
                    f"candidates[{i}] must be a dict. "
                    "Rebuild the full 10-candidate list and ensure every candidate is a complete dict."
                )

            for k in required_keys:
                if k not in item:
                    return _error(
                        f"candidates[{i}] missing required key '{k}'. "
                        "Every candidate must include plot_id, title, description, plot_type, data_requirements, axes, and rationale."
                    )

            plot_id = item.get("plot_id")
            title = item.get("title")
            description = item.get("description")
            data_requirements = item.get("data_requirements")
            axes = item.get("axes")
            rationale = item.get("rationale")
            plot_type = item.get("plot_type")

            if not isinstance(plot_id, str) or not plot_id.strip():
                return _error(f"candidates[{i}].plot_id must be a non-empty string.")
            plot_id_s = plot_id.strip()
            if plot_id_s in seen_ids:
                return _error(
                    f"Duplicate plot_id found: {plot_id_s}. "
                    "Use exactly one stable candidate per required plot_id."
                )
            seen_ids.add(plot_id_s)

            expected_plot_id = expected_ids[i]
            if plot_id_s != expected_plot_id:
                return _error(
                    f"candidates[{i}].plot_id must be exactly '{expected_plot_id}', got '{plot_id_s}'. "
                    "Use stable ids in exact order: plot_1 through plot_10."
                )

            if not isinstance(title, str) or not title.strip():
                return _error(f"candidates[{i}].title must be a non-empty string.")
            if not isinstance(description, str) or not description.strip():
                return _error(f"candidates[{i}].description must be a non-empty string.")
            if not isinstance(rationale, str) or not rationale.strip():
                return _error(f"candidates[{i}].rationale must be a non-empty string.")

            if not isinstance(data_requirements, list) or not data_requirements or not all(
                isinstance(x, str) and x.strip() for x in data_requirements
            ):
                return _error(
                    f"candidates[{i}].data_requirements must be a non-empty list of non-empty strings."
                )

            if not isinstance(axes, dict) or not axes:
                return _error(f"candidates[{i}].axes must be a non-empty dict.")

            raw_strings_to_check = []
            raw_strings_to_check.extend([x.strip() for x in data_requirements if isinstance(x, str)])
            for axis_key in ("x", "y", "z", "intensity"):
                axis_value = axes.get(axis_key)
                if isinstance(axis_value, str):
                    raw_strings_to_check.append(axis_value.strip())

            bad_aliases_used = sorted({x for x in raw_strings_to_check if x in forbidden_derived_aliases})
            if bad_aliases_used:
                return _error(
                    f"candidates[{i}] uses forbidden derived or aggregate fields {bad_aliases_used}. "
                    "Use only direct per-material numeric fields available in this run: band_gap, density, energy_above_hull."
                )

            try:
                candidate_payload = {
                    "plot_id": plot_id,
                    "title": title,
                    "plot_type": plot_type,
                    "data_requirements": data_requirements,
                    "axes": axes,
                }
                candidate_model = PlotCandidate(**candidate_payload)
                candidate_model.validate_structure()
            except Exception as e:
                return _error(f"candidates[{i}] invalid structure: {e}")

            normalized_axes = candidate_model.axes.model_dump()
            normalized_requirements = [x.strip() for x in candidate_model.data_requirements]
            normalized_plot_type = candidate_model.plot_type.value

            if normalized_plot_type not in {"scatter_2d", "histogram", "scatter_3d"}:
                return _error(
                    f"candidates[{i}] uses unsupported plot_type '{normalized_plot_type}' for this run. "
                    "Use only scatter_2d, histogram, or scatter_3d."
                )

            uses_derived_field = any(x in allowed_derived_fields for x in normalized_requirements)
            uses_derived_axis = any(
                normalized_axes.get(axis_key) in allowed_derived_fields
                for axis_key in ("x", "y", "z", "intensity")
            )
            uses_count_axis = normalized_plot_type == "bar" and normalized_axes.get("y") == "count"

            is_derived_plot = uses_derived_field or uses_derived_axis or uses_count_axis

            if is_derived_plot:
                derived_plot_count += 1
                if normalized_plot_type == "bar":
                    derived_bar_count += 1
            else:
                direct_plot_count += 1

            for field_name in normalized_requirements:
                if field_name in available_direct_fields:
                    continue
                return _error(
                    f"candidates[{i}].data_requirements contains unavailable field '{field_name}'. "
                    f"Available direct fields for this run: {sorted(available_direct_fields)}."
                )

            axis_fields_to_check = []
            for axis_key in ("x", "y", "z", "intensity"):
                axis_value = normalized_axes.get(axis_key)
                if axis_value is None:
                    continue
                axis_fields_to_check.append((axis_key, axis_value))

            for axis_key, axis_value in axis_fields_to_check:
                if axis_value in available_direct_fields:
                    pass
                else:
                    return _error(
                        f"candidates[{i}].axes.{axis_key} uses unavailable field '{axis_value}'. "
                        f"Available direct fields for this run: {sorted(available_direct_fields)}."
                    )

            for axis_key, axis_value in axis_fields_to_check:
                if axis_value not in normalized_requirements:
                    return _error(
                        f"candidates[{i}].axes.{axis_key}='{axis_value}' must appear in data_requirements. "
                        "Every non-null axis field must also appear in data_requirements under the exact same name."
                    )

            normalized.append(
                {
                    "plot_id": candidate_model.plot_id.strip(),
                    "title": candidate_model.title.strip(),
                    "description": description.strip(),
                    "data_requirements": normalized_requirements,
                    "axes": normalized_axes,
                    "rationale": rationale.strip(),
                    "plot_type": normalized_plot_type,
                }
            )

        if derived_plot_count > 0:
            return _error(
                f"Derived aggregate plots are not allowed in this run. Got {derived_plot_count}. "
                "Use only direct per-material plots built from band_gap, density, and energy_above_hull."
            )

        if derived_bar_count > 0:
            return _error(
                f"Bar plots are not allowed in this run. Got {derived_bar_count}. "
                "Replace them with direct scatter_2d, histogram, or scatter_3d candidates."
            )

        if direct_plot_count != 10:
            return _error(
                f"All 10 plot candidates must be direct per-material plots. Got {direct_plot_count} direct plots."
            )

    except ValueError as e:
        return _error(str(e))

    # Validate non-numeric fields
    NON_NUMERIC_FIELDS = {"elements", "formula", "formula_pretty", "material_id", "spacegroup", "symmetry", "structure", "composition", "is_stable"}
    for idx, plot in enumerate(normalized):
        plot_type = getattr(plot, "plot_type", None) or (plot.get("plot_type") if isinstance(plot, dict) else None)
        if plot_type in ("scatter_2d", "scatter_3d", "histogram"):
            axes = getattr(plot, "axes", None) or (plot.get("axes") if isinstance(plot, dict) else None)
            if axes:
                x = getattr(axes, "x", None) or (axes.get("x") if isinstance(axes, dict) else None)
                y = getattr(axes, "y", None) or (axes.get("y") if isinstance(axes, dict) else None)
                z = getattr(axes, "z", None) or (axes.get("z") if isinstance(axes, dict) else None)
                intensity = getattr(axes, "intensity", None) or (axes.get("intensity") if isinstance(axes, dict) else None)
                for field_name, field_val in [("x", x), ("y", y), ("z", z), ("intensity", intensity)]:
                    if field_val and field_val in NON_NUMERIC_FIELDS:
                        return _error(
                            f"REJECTED: plot_{idx+1} uses non-numeric field '{field_val}' as {field_name}-axis. "
                            f"Fields {sorted(NON_NUMERIC_FIELDS)} cannot be used as numeric axes. Use a numeric field instead."
                        )

    context_variables["plot_candidates"] = normalized
    context_variables["plot_candidates_error"] = ""
    context_variables["plot_suggest_status"] = "completed"
    context_variables["next_agent"] = next_agent

    message = {
        "plot_candidates_count": len(normalized),
        "plot_candidates": normalized,
    }

    return ReplyResult(
        message=json.dumps(message, indent=2, ensure_ascii=False),
        target=AgentNameTarget(next_agent),
        context_variables=context_variables,
    )

# Tool — plot_selector_tool

def plot_selector_tool(
    selected_plot_ids: Annotated[List[str], "List of selected plot_id values (must select exactly 5)."],
    selection_notes: Annotated[str, "Short notes explaining why these plots were selected."],
    context_variables: ContextVariables,
) -> ReplyResult:
    agent_name = "AgentPlotSelector"

    if context_variables.get("plot_suggest_status") == "completed" and context_variables.get("plot_selected"):
        return ReplyResult(
            message="Plot selection already completed. Ignoring outdated selector call.",
            target=AgentNameTarget(context_variables.get("next_agent", "AgentCoder")),
            context_variables=context_variables,
        )

    plan = context_variables.get("plan", {})
    routing = plan.get("routing", {}) if isinstance(plan, dict) else {}
    next_agent = routing.get("after_plot_select", None)
    if not isinstance(next_agent, str) or not next_agent.strip():
        next_agent = "AgentCoder"

    try:
        if not isinstance(selected_plot_ids, list) or len(selected_plot_ids) == 0:
            raise ValueError("selected_plot_ids must be a non-empty list.")
        if not all(isinstance(x, str) and x.strip() for x in selected_plot_ids):
            raise ValueError("selected_plot_ids must contain non-empty strings only.")
        if not isinstance(selection_notes, str) or not selection_notes.strip():
            raise ValueError("selection_notes must be a non-empty string.")
    except ValueError as e:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in plot_selector_tool: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    try:
        candidates = context_variables.get("plot_candidates", None)
        if not isinstance(candidates, list) or len(candidates) == 0:
            raise ValueError("Missing or invalid context_variables['plot_candidates'] (must be a non-empty list).")

        if len(candidates) != 10:
            raise ValueError(f"Expected exactly 10 plot_candidates, got {len(candidates)}.")

        retrieval = plan.get("retrieval", {}) if isinstance(plan, dict) else {}
        retrieval_fields = retrieval.get("fields", []) if isinstance(retrieval, dict) else []
        if not isinstance(retrieval_fields, list):
            retrieval_fields = []

        available_direct_fields = {x.strip() for x in retrieval_fields if isinstance(x, str) and x.strip()}

        analysis_summary = context_variables.get("analysis_summary", {})
        derived_fields_available = set()
        if isinstance(analysis_summary, dict):
            patterns = analysis_summary.get("patterns", {})
            if isinstance(patterns, dict):
                if "stability_bins" in patterns:
                    derived_fields_available.add("stability_bins")
                if "band_gap_bins" in patterns:
                    derived_fields_available.add("band_gap_bins")

        allowed_derived_fields = {"stability_bins", "band_gap_bins"}
        allowed_axis_count_labels = {"count", "number_of_materials", "materials_count"}

        id_to_candidate = {}
        for c in candidates:
            if not isinstance(c, dict):
                raise ValueError("Each plot candidate must be a dict.")

            pid = c.get("plot_id")
            title = c.get("title")
            description = c.get("description")
            data_requirements = c.get("data_requirements")
            axes = c.get("axes")
            plot_type = c.get("plot_type")

            if not isinstance(pid, str) or not pid.strip():
                raise ValueError("Each plot candidate must have a non-empty 'plot_id'.")
            if not isinstance(title, str) or not title.strip():
                raise ValueError(f"Plot candidate '{pid}' must have a non-empty 'title'.")
            if not isinstance(description, str) or not description.strip():
                raise ValueError(f"Plot candidate '{pid}' must have a non-empty 'description'.")
            if not isinstance(data_requirements, list) or not data_requirements or not all(
                isinstance(x, str) and x.strip() for x in data_requirements
            ):
                raise ValueError(f"Plot candidate '{pid}' must have valid data_requirements.")
            if not isinstance(axes, dict) or not axes:
                raise ValueError(f"Plot candidate '{pid}' must have valid axes.")
            if not isinstance(plot_type, str) or not plot_type.strip():
                raise ValueError(f"Plot candidate '{pid}' must have a valid plot_type.")
            if pid in id_to_candidate:
                raise ValueError(f"Duplicate plot_id found: {pid}")

            id_to_candidate[pid] = c

        plotting = plan.get("plotting", {}) if isinstance(plan, dict) else {}
        n_selected = plotting.get("n_selected", 5)
        if not isinstance(n_selected, int) or n_selected <= 0:
            n_selected = 5

        ids_normalized = [x.strip() for x in selected_plot_ids]
        if len(set(ids_normalized)) != len(ids_normalized):
            raise ValueError("Duplicate plot_id values are not allowed.")
        if len(ids_normalized) != n_selected:
            raise ValueError(f"Must select exactly {n_selected} plot_id values.")

        missing = [pid for pid in ids_normalized if pid not in id_to_candidate]
        if missing:
            raise ValueError(f"Selected plot_id not found in candidates: {missing}")

        selected = [id_to_candidate[pid] for pid in ids_normalized]

        robust_plot_types = {"scatter_2d", "histogram"}
        fragile_plot_types = {"scatter_3d", "heatmap_2d", "bar"}

        for i, item in enumerate(selected):
            pid = item["plot_id"]
            data_requirements = [x.strip() for x in item.get("data_requirements", [])]
            axes = item.get("axes", {})
            plot_type = item.get("plot_type", "")

            for field_name in data_requirements:
                if field_name in available_direct_fields:
                    continue
                if field_name in allowed_derived_fields and field_name in derived_fields_available:
                    continue
                raise ValueError(
                    f"Selected plot '{pid}' uses unavailable data_requirement '{field_name}'. "
                    f"Available direct fields: {sorted(available_direct_fields)}. "
                    f"Available derived fields: {sorted(derived_fields_available)}."
                )

            for axis_key in ("x", "y", "z", "intensity"):
                axis_value = axes.get(axis_key)
                if axis_value is None:
                    continue

                if axis_value in available_direct_fields:
                    pass
                elif axis_value in allowed_derived_fields and axis_value in derived_fields_available:
                    pass
                elif plot_type == "bar" and axis_key == "y" and axis_value in allowed_axis_count_labels:
                    pass
                else:
                    raise ValueError(
                        f"Selected plot '{pid}' uses unavailable axes.{axis_key}='{axis_value}'. "
                        f"Available direct fields: {sorted(available_direct_fields)}. "
                        f"Available derived fields: {sorted(derived_fields_available)}."
                    )

                if not (plot_type == "bar" and axis_key == "y" and axis_value in allowed_axis_count_labels):
                    if axis_value not in data_requirements:
                        raise ValueError(
                            f"Selected plot '{pid}' has axes.{axis_key}='{axis_value}' not present in data_requirements."
                        )

            if plot_type not in robust_plot_types and plot_type not in fragile_plot_types:
                raise ValueError(
                    f"Selected plot '{pid}' has unsupported plot_type '{plot_type}' for the current coder."
                )

            if plot_type == "scatter_3d":
                if axes.get("type") != "3d":
                    raise ValueError(f"Selected plot '{pid}' must have axes.type='3d' for scatter_3d.")
                required_axes = {"x", "y", "z"}
                present_axes = {k for k in required_axes if axes.get(k) is not None}
                if present_axes != required_axes:
                    raise ValueError(f"Selected plot '{pid}' must define x, y, and z axes for scatter_3d.")
                if len(data_requirements) != 3:
                    raise ValueError(
                        f"Selected plot '{pid}' is too fragile for the current coder because scatter_3d must use exactly 3 direct data_requirements."
                    )
                if any(field in allowed_derived_fields for field in data_requirements):
                    raise ValueError(
                        f"Selected plot '{pid}' is too fragile for the current coder because scatter_3d must use direct per-material fields only."
                    )

            if plot_type == "heatmap_2d":
                if axes.get("type") != "heatmap":
                    raise ValueError(f"Selected plot '{pid}' must have axes.type='heatmap' for heatmap_2d.")
                if axes.get("x") is None or axes.get("y") is None or axes.get("intensity") is None:
                    raise ValueError(
                        f"Selected plot '{pid}' must define x, y, and intensity axes for heatmap_2d."
                    )
                if len(data_requirements) != 3:
                    raise ValueError(
                        f"Selected plot '{pid}' is too fragile for the current coder because heatmap_2d must use exactly 3 direct data_requirements."
                    )
                if any(field in allowed_derived_fields for field in data_requirements):
                    raise ValueError(
                        f"Selected plot '{pid}' is too fragile for the current coder because heatmap_2d must use direct per-material fields only."
                    )
                if "volume" in data_requirements:
                    raise ValueError(
                        f"Selected plot '{pid}' is too fragile for the current coder because heatmap_2d using 'volume' is not currently robust."
                    )

            if plot_type == "bar":
                if any(field in allowed_derived_fields for field in data_requirements):
                    raise ValueError(
                        f"Selected plot '{pid}' is too fragile for the current coder because derived bar plots are not currently robust enough."
                    )
                if "elements" in data_requirements:
                    raise ValueError(
                        f"Selected plot '{pid}' is too fragile for the current coder because element-based categorical aggregation is not currently robust."
                    )

    except ValueError as e:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in plot_selector_tool: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    context_variables["plot_selected"] = selected
    context_variables["plot_selection_notes"] = selection_notes.strip()
    context_variables["next_agent"] = next_agent

    message = {
        "plot_selected_ids": ids_normalized,
        "plot_selected": selected,
        "selection_notes": context_variables["plot_selection_notes"],
    }

    return ReplyResult(
        message=json.dumps(message, indent=2, ensure_ascii=False),
        target=AgentNameTarget(next_agent),
        context_variables=context_variables,
    )

# Tool — python_coder

project_folder = os.environ.get("PROJECT_FOLDER", os.path.abspath("ag2_project"))
os.makedirs(project_folder, exist_ok=True)
os.environ["PROJECT_FOLDER"] = project_folder


class PythonCode(BaseModel):
    code: str = Field(..., description="Full python code executed by the coder agent")


def _safe_filename(name: str) -> str:
    name = os.path.basename((name or "").strip())
    if len(name) == 0:
        return "script.py"

    keep = []
    for ch in name:
        if ch.isalnum() or ch in {".", "_", "-"}:
            keep.append(ch)
    name = "".join(keep)

    if len(name) == 0:
        name = "script.py"
    if not name.endswith(".py"):
        name = name + ".py"
    if len(name) > 120:
        name = name[-120:]
    return name


def _extract_plot_id_from_png_name(filename: str) -> str:
    if not isinstance(filename, str):
        return ""
    match = re.match(r"^(plot_\d+)_", os.path.basename(filename))
    if match:
        return match.group(1)
    return ""


def _code_has_plot_branch(code_str: str, plot_type: str) -> bool:
    single = f"plot_type == '{plot_type}'"
    double = f'plot_type == "{plot_type}"'
    return single in code_str or double in code_str


def _has_empty_elif_block(code_str: str) -> bool:
    lines = code_str.splitlines()
    for i, line in enumerate(lines[:-1]):
        stripped = line.strip()
        if stripped.startswith("elif ") and stripped.endswith(":"):
            next_line = lines[i + 1]
            if len(next_line.strip()) == 0:
                return True
            if len(next_line) - len(next_line.lstrip()) <= len(line) - len(line.lstrip()):
                return True
    return False


def _count_plot_type_branches(code_str: str, plot_type: str) -> int:
    single = f"plot_type == '{plot_type}'"
    double = f'plot_type == "{plot_type}"'
    return code_str.count(single) + code_str.count(double)


def _has_fragile_histogram_pattern(code_str: str) -> bool:
    if "plot_type == 'histogram'" not in code_str and 'plot_type == "histogram"' not in code_str:
        return False

    if "plt.hist(" not in code_str:
        return True

    fragile_patterns = [
        "if 'band_gap' in plot_spec['data_requirements']",
        'if "band_gap" in plot_spec["data_requirements"]',
        "if 'density' in plot_spec['data_requirements']",
        'if "density" in plot_spec["data_requirements"]',
        "if 'energy_above_hull' in plot_spec['data_requirements']",
        'if "energy_above_hull" in plot_spec["data_requirements"]',
        "x_data, y_data, z_data, intensity_data = [], [], [], []",
        "y_data = []",
        "z_data = []",
        "intensity_data = []",
    ]

    found_fragile_count = sum(1 for pattern in fragile_patterns if pattern in code_str)
    return found_fragile_count >= 2


def _has_inconsistent_manifest_naming_pattern(code_str: str) -> bool:
    bad_patterns = [
        'filename = f"{title',
        "filename = f'{title",
        'name": title.lower(',
        "name': title.lower(",
        'path": os.path.join("plots", title.lower(',
        "path': os.path.join('plots', title.lower(",
        'path": "plots/" + title.lower(',
        "path': 'plots/' + title.lower(",
        'filename = title.lower(',
        "filename = title.lower(",
    ]
    return any(pattern in code_str for pattern in bad_patterns)


def _has_manifest_comprehension_pattern(code_str: str) -> bool:
    bad_patterns = [
        "plots_v1 = [{",
        "plots_v2 = [{",
        '"plots_v1": [{',
        '"plots_v2": [{',
        "'plots_v1': [{",
        "'plots_v2': [{",
        "for plot_spec in plot_selected]",
        "for spec in plot_selected]",
        "for plot in plot_selected]",
    ]
    return any(pattern in code_str for pattern in bad_patterns)


def _has_stable_placeholder_pattern(code_str: str) -> bool:
    required_markers = [
        "Insufficient data",
        "plt.savefig(out_path)",
        "os.path.exists(out_path)",
    ]
    return all(marker in code_str for marker in required_markers)


def _is_overly_complex_plot_script(code_str: str) -> bool:
    metrics = 0

    if code_str.count("try:") >= 3:
        metrics += 1
    if code_str.count("except ") >= 3:
        metrics += 1
    if code_str.count("def ") >= 5:
        metrics += 1
    if code_str.count("for plot_spec in plot_selected") >= 2:
        metrics += 1
    if code_str.count("for spec in plot_selected") >= 2:
        metrics += 1
    if code_str.count("for plot in plot_selected") >= 2:
        metrics += 1
    if code_str.count("plt.figure(") >= 8:
        metrics += 1
    if len(code_str.splitlines()) >= 260:
        metrics += 1

    return metrics >= 2


def python_coder_tool(
    code: Annotated[str, "A single block of Python code to execute."],
    file_name: Annotated[str, "Name of the code file to store the executed script, e.g., 'script.py'"],
    context_variables: ContextVariables,
) -> ReplyResult:
    agent_name = "AgentCoder"

    _MAX_CODER_RETRIES = 5
    _coder_retries = int(context_variables.get("coder_retries", 0))

    plan = context_variables.get("plan", {})
    routing = plan.get("routing", {}) if isinstance(plan, dict) else {}

    next_agent_v1 = routing.get("after_code", None)
    if not isinstance(next_agent_v1, str) or not next_agent_v1.strip():
        next_agent_v1 = "Human"

    next_agent_v2 = routing.get("after_plot_validate", None)
    if not isinstance(next_agent_v2, str) or not next_agent_v2.strip():
        next_agent_v2 = next_agent_v1

    try:
        if not isinstance(code, str) or len(code.strip()) == 0:
            raise ValueError("The 'code' parameter must be a non-empty string.")
        if not isinstance(file_name, str) or len(file_name.strip()) == 0:
            raise ValueError("The 'file_name' parameter must be a non-empty string.")
    except ValueError as e:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in python_coder_tool: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    safe_name = _safe_filename(file_name)

    try:
        os.environ["PLOT_SELECTED_JSON"] = json.dumps(
            context_variables.get("plot_selected", []),
            ensure_ascii=False,
            default=str,
        )
    except Exception:
        os.environ["PLOT_SELECTED_JSON"] = "[]"

    selected_specs = context_variables.get("plot_selected", [])
    plots_to_regenerate_raw = context_variables.get("plots_to_regenerate", [])

    plots_to_regenerate = []
    if isinstance(plots_to_regenerate_raw, list):
        for x in plots_to_regenerate_raw:
            if isinstance(x, str) and x.strip():
                plots_to_regenerate.append(x.strip())

    if len(set(plots_to_regenerate)) != len(plots_to_regenerate):
        plots_to_regenerate = list(dict.fromkeys(plots_to_regenerate))

    is_v2 = len(plots_to_regenerate) > 0

    code_stripped = code.strip()

    forbidden_plot_selected_patterns = [
        "plot_selected =",
        "plot_selected=",
        "plot_selected = [",
        "plot_selected=[",
        "plot_selected = list(",
        "plot_selected=list(",
        "plot_selected = json.loads(",
        "plot_selected=json.loads(",
    ]

    if "plot_selected" not in code_stripped:
        context_variables["plots_status"] = "v2_failed" if is_v2 else "v1_failed"
        _coder_retries += 1
        context_variables["coder_retries"] = _coder_retries
        if _coder_retries > _MAX_CODER_RETRIES:
            return ReplyResult(
                message=(
                    f"Error in python_coder_tool: AgentCoder exceeded {_MAX_CODER_RETRIES} retries. "
                    "Handing off to Human for manual intervention."
                ),
                target=AgentNameTarget("Human"),
                context_variables=context_variables,
            )
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=(
                "Error in python_coder_tool: generated code does not reference 'plot_selected'. "
                "Plots must be generated by iterating over the runtime plot_selected variable."
            ),
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    if any(pattern in code_stripped for pattern in forbidden_plot_selected_patterns):
        context_variables["plots_status"] = "v2_failed" if is_v2 else "v1_failed"
        _coder_retries += 1
        context_variables["coder_retries"] = _coder_retries
        if _coder_retries > _MAX_CODER_RETRIES:
            return ReplyResult(
                message=(
                    f"Error in python_coder_tool: AgentCoder exceeded {_MAX_CODER_RETRIES} retries. "
                    "Handing off to Human for manual intervention."
                ),
                target=AgentNameTarget("Human"),
                context_variables=context_variables,
            )
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=(
                "Error in python_coder_tool: generated code redefines 'plot_selected'. "
                "The runtime variable 'plot_selected' is the single source of truth and must be used directly."
            ),
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

        if "plot_selected[0]" in code_stripped:
            context_variables["plots_status"] = "v2_failed" if is_v2 else "v1_failed"
            _coder_retries += 1
            context_variables["coder_retries"] = _coder_retries
            if _coder_retries > _MAX_CODER_RETRIES:
                return ReplyResult(
                    message=(
                        f"Error in python_coder_tool: AgentCoder exceeded {_MAX_CODER_RETRIES} retries. "
                        "Handing off to Human for manual intervention."
                    ),
                    target=AgentNameTarget("Human"),
                    context_variables=context_variables,
                )
            context_variables["next_agent"] = agent_name
            return ReplyResult(
                message=(
                    "Error in python_coder_tool: generated code uses 'plot_selected[0]' "
                    "inside the loop. Always use the current iteration variable "
                    "(e.g. axes['x'], axes['y']) instead of indexing plot_selected directly."
                ),
                target=AgentNameTarget(agent_name),
                context_variables=context_variables,
            )

    selected_plot_types = []
    if isinstance(selected_specs, list):
        for spec in selected_specs:
            if isinstance(spec, dict):
                plot_type = spec.get("plot_type")
                if isinstance(plot_type, str) and plot_type.strip():
                    selected_plot_types.append(plot_type.strip())

    for plot_type in sorted(set(selected_plot_types)):
        if not _code_has_plot_branch(code_stripped, plot_type):
            context_variables["plots_status"] = "v2_failed" if is_v2 else "v1_failed"
            context_variables["next_agent"] = agent_name
            return ReplyResult(
                message=(
                    f"Error in python_coder_tool: generated code does not contain a clear branch for "
                    f"plot_type '{plot_type}'."
                ),
                target=AgentNameTarget(agent_name),
                context_variables=context_variables,
            )

    if _has_empty_elif_block(code_stripped):
        context_variables["plots_status"] = "v2_failed" if is_v2 else "v1_failed"
        _coder_retries += 1
        context_variables["coder_retries"] = _coder_retries
        if _coder_retries > _MAX_CODER_RETRIES:
            return ReplyResult(
                message=(
                    f"Error in python_coder_tool: AgentCoder exceeded {_MAX_CODER_RETRIES} retries. "
                    "Handing off to Human for manual intervention."
                ),
                target=AgentNameTarget("Human"),
                context_variables=context_variables,
            )
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=(
                "Error in python_coder_tool: generated code contains an incomplete if/elif plotting branch. "
                "Every if / elif / else branch must have a full indented body. "
                "Use one stable plotting chain with complete branches for scatter_2d, histogram, bar, heatmap_2d, scatter_3d, and else."
            ),
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    for plot_type in sorted(set(selected_plot_types)):
        if _count_plot_type_branches(code_stripped, plot_type) > 1:
            context_variables["plots_status"] = "v2_failed" if is_v2 else "v1_failed"
            context_variables["next_agent"] = agent_name
            return ReplyResult(
                message=(
                    f"Error in python_coder_tool: generated code contains multiple branches for plot_type '{plot_type}'. "
                    "Use one single stable if/elif chain with exactly one branch per plot type."
                ),
                target=AgentNameTarget(agent_name),
                context_variables=context_variables,
            )

    if "scatter_3d" in selected_plot_types:
        fragile_3d_patterns = [
            "plt.gca(projection='3d')",
            'plt.gca(projection="3d")',
            "plt.figure().add_subplot(",
        ]
        if any(pattern in code_stripped for pattern in fragile_3d_patterns):
            context_variables["plots_status"] = "v2_failed" if is_v2 else "v1_failed"
            context_variables["next_agent"] = agent_name
            return ReplyResult(
                message=(
                    "Error in python_coder_tool: generated code uses a fragile 3D plotting pattern. "
                    "Use only: fig = plt.figure() and ax = fig.add_subplot(111, projection='3d') inside the scatter_3d branch."
                ),
                target=AgentNameTarget(agent_name),
                context_variables=context_variables,
            )

        if "fig = plt.figure()" not in code_stripped or "ax = fig.add_subplot(111, projection='3d')" not in code_stripped:
            if 'ax = fig.add_subplot(111, projection="3d")' not in code_stripped:
                context_variables["plots_status"] = "v2_failed" if is_v2 else "v1_failed"
                context_variables["next_agent"] = agent_name
                return ReplyResult(
                    message=(
                        "Error in python_coder_tool: scatter_3d must use an explicit dedicated 3D figure pattern "
                        "with fig = plt.figure() and ax = fig.add_subplot(111, projection='3d')."
                    ),
                    target=AgentNameTarget(agent_name),
                    context_variables=context_variables,
                )

    if "histogram" in selected_plot_types and _has_fragile_histogram_pattern(code_stripped):
        context_variables["plots_status"] = "v2_failed" if is_v2 else "v1_failed"
        _coder_retries += 1
        context_variables["coder_retries"] = _coder_retries
        if _coder_retries > _MAX_CODER_RETRIES:
            return ReplyResult(
                message=(
                    f"Error in python_coder_tool: AgentCoder exceeded {_MAX_CODER_RETRIES} retries. "
                    "Handing off to Human for manual intervention."
                ),
                target=AgentNameTarget("Human"),
                context_variables=context_variables,
            )
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=(
                "Error in python_coder_tool: histogram handling is too fragile. "
                "Histogram must have its own independent branch, must use plt.hist(...), "
                "and must not depend on a shared scatter-style data preparation path."
            ),
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    if _has_inconsistent_manifest_naming_pattern(code_stripped):
        context_variables["plots_status"] = "v2_failed" if is_v2 else "v1_failed"
        _coder_retries += 1
        context_variables["coder_retries"] = _coder_retries
        if _coder_retries > _MAX_CODER_RETRIES:
            return ReplyResult(
                message=(
                    f"Error in python_coder_tool: AgentCoder exceeded {_MAX_CODER_RETRIES} retries. "
                    "Handing off to Human for manual intervention."
                ),
                target=AgentNameTarget("Human"),
                context_variables=context_variables,
            )
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=(
                "Error in python_coder_tool: manifest naming is inconsistent with the plot filename contract. "
                "Use one single filename rule only: filename = f\"{plot_id}_{slug}.png\" "
                "and make the manifest name/path match the exact saved file."
            ),
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    if _has_manifest_comprehension_pattern(code_stripped):
        context_variables["plots_status"] = "v2_failed" if is_v2 else "v1_failed"
        _coder_retries += 1
        context_variables["coder_retries"] = _coder_retries
        if _coder_retries > _MAX_CODER_RETRIES:
            return ReplyResult(
                message=(
                    f"Error in python_coder_tool: AgentCoder exceeded {_MAX_CODER_RETRIES} retries. "
                    "Handing off to Human for manual intervention."
                ),
                target=AgentNameTarget("Human"),
                context_variables=context_variables,
            )
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=(
                "Error in python_coder_tool: manifest construction is too complex or fragile. "
                "Do not use list comprehensions or one-line manifest builders. "
                "Build manifest entries with an explicit loop and explicit local variables."
            ),
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    if not _has_stable_placeholder_pattern(code_stripped):
        context_variables["plots_status"] = "v2_failed" if is_v2 else "v1_failed"
        _coder_retries += 1
        context_variables["coder_retries"] = _coder_retries
        if _coder_retries > _MAX_CODER_RETRIES:
            return ReplyResult(
                message=(
                    f"Error in python_coder_tool: AgentCoder exceeded {_MAX_CODER_RETRIES} retries. "
                    "Handing off to Human for manual intervention."
                ),
                target=AgentNameTarget("Human"),
                context_variables=context_variables,
            )
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=(
                "Error in python_coder_tool: generated code does not show a stable placeholder strategy. "
                "Use one simple placeholder path with the exact text 'Insufficient data', "
                "the normal out_path saveflow, and the same naming contract as regular plots."
            ),
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    if _is_overly_complex_plot_script(code_stripped):
        context_variables["plots_status"] = "v2_failed" if is_v2 else "v1_failed"
        _coder_retries += 1
        context_variables["coder_retries"] = _coder_retries
        if _coder_retries > _MAX_CODER_RETRIES:
            return ReplyResult(
                message=(
                    f"Error in python_coder_tool: AgentCoder exceeded {_MAX_CODER_RETRIES} retries. "
                    "Handing off to Human for manual intervention."
                ),
                target=AgentNameTarget("Human"),
                context_variables=context_variables,
            )
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=(
                "Error in python_coder_tool: generated plotting script is unnecessarily complex. "
                "Reduce complexity and use one stable explicit implementation per plot type, "
                "one explicit placeholder path, and one explicit manifest loop."
            ),
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    context_variables["plots_status"] = "v2_running" if is_v2 else "v1_running"

    expected_plot_ids = []
    if isinstance(selected_specs, list):
        for s in selected_specs:
            if isinstance(s, dict):
                pid = s.get("plot_id")
                if isinstance(pid, str) and pid.strip():
                    expected_plot_ids.append(pid.strip())

    expects_v1_plots = (not is_v2) and isinstance(selected_specs, list) and len(selected_specs) == 5

    if expects_v1_plots:
        if len(expected_plot_ids) != 5:
            context_variables["plots_status"] = "v1_failed"
            context_variables["next_agent"] = agent_name
            return ReplyResult(
                message=(
                    "Error in python_coder_tool: plot_selected must contain exactly 5 items with non-empty 'plot_id' "
                    "values before generating plots."
                ),
                target=AgentNameTarget(agent_name),
                context_variables=context_variables,
            )
        if len(set(expected_plot_ids)) != len(expected_plot_ids):
            context_variables["plots_status"] = "v1_failed"
            context_variables["next_agent"] = agent_name
            return ReplyResult(
                message="Error in python_coder_tool: plot_selected contains duplicate plot_id values.",
                target=AgentNameTarget(agent_name),
                context_variables=context_variables,
            )

    if expects_v1_plots:
        try:
            plots_dir_v1_pre = os.path.join(project_folder, "plots")
            os.makedirs(plots_dir_v1_pre, exist_ok=True)
            for fn in os.listdir(plots_dir_v1_pre):
                if fn.lower().endswith(".png"):
                    try:
                        os.remove(os.path.join(plots_dir_v1_pre, fn))
                    except Exception:
                        pass
        except Exception:
            pass

    if is_v2:
        try:
            plots_dir_v2_pre = os.path.join(project_folder, "plots_v2")
            os.makedirs(plots_dir_v2_pre, exist_ok=True)
            for fn in os.listdir(plots_dir_v2_pre):
                if not fn.lower().endswith(".png"):
                    continue
                pid = _extract_plot_id_from_png_name(fn)
                if pid in plots_to_regenerate:
                    try:
                        os.remove(os.path.join(plots_dir_v2_pre, fn))
                    except Exception:
                        pass
        except Exception:
            pass

    try:
        os.environ["PLOTS_TO_REGENERATE_JSON"] = json.dumps(
            plots_to_regenerate,
            ensure_ascii=False,
            default=str,
        )
    except Exception:
        os.environ["PLOTS_TO_REGENERATE_JSON"] = "[]"

    runtime_prelude = (
        "import os\n"
        "import json\n"
        "PROJECT_FOLDER = os.environ.get('PROJECT_FOLDER', 'ag2_project')\n"
        "os.makedirs(PROJECT_FOLDER, exist_ok=True)\n"
        "os.chdir(PROJECT_FOLDER)\n"
        "\n"
        "PLOTS_DIR_V1 = os.path.join(PROJECT_FOLDER, 'plots')\n"
        "PLOTS_DIR_V2 = os.path.join(PROJECT_FOLDER, 'plots_v2')\n"
        "os.makedirs(PLOTS_DIR_V1, exist_ok=True)\n"
        "os.makedirs(PLOTS_DIR_V2, exist_ok=True)\n"
        "\n"
        "PLOT_SELECTED_JSON = os.environ.get('PLOT_SELECTED_JSON', '[]')\n"
        "try:\n"
        "    plot_selected = json.loads(PLOT_SELECTED_JSON)\n"
        "except Exception:\n"
        "    plot_selected = []\n"
        "\n"
        "PLOTS_TO_REGENERATE_JSON = os.environ.get('PLOTS_TO_REGENERATE_JSON', '[]')\n"
        "try:\n"
        "    plots_to_regenerate = json.loads(PLOTS_TO_REGENERATE_JSON)\n"
        "except Exception:\n"
        "    plots_to_regenerate = []\n"
        "\n"
        "def _ensure_materials_data_json():\n"
        "    if os.path.exists('materials_data.json'):\n"
        "        return\n"
        "    if not os.path.exists('final_conclusion.json'):\n"
        "        return\n"
        "    try:\n"
        "        with open('final_conclusion.json', 'r', encoding='utf-8') as f:\n"
        "            payload = json.load(f)\n"
        "    except Exception:\n"
        "        return\n"
        "    mp_results = payload.get('mp_results')\n"
        "    analysis_summary = payload.get('analysis_summary')\n"
        "    final_conclusion = payload.get('final_conclusion')\n"
        "    if mp_results is None:\n"
        "        return\n"
        "    with open('materials_data.json', 'w', encoding='utf-8') as f:\n"
        "        json.dump(\n"
        "            {\n"
        "                'mp_results': mp_results,\n"
        "                'analysis_summary': analysis_summary,\n"
        "                'final_conclusion': final_conclusion,\n"
        "            },\n"
        "            f,\n"
        "            indent=2,\n"
        "            ensure_ascii=False,\n"
        "            default=str,\n"
        "        )\n"
        "\n"
        "_ensure_materials_data_json()\n"
    )

    code_sanitized = code.replace("\t", "    ").strip("\n")
    sanitized_lines = []
    for line in code_sanitized.splitlines():
        stripped = line.lstrip()
        if stripped.startswith(("def ", "class ", "import ", "from ")):
            sanitized_lines.append(stripped)
        else:
            sanitized_lines.append(line.rstrip())
    code_sanitized = "\n".join(sanitized_lines).strip()

    full_code = "#!/usr/bin/env python3\n" + runtime_prelude + "\n" + code_sanitized + "\n"

    try:
        compile(full_code, safe_name, "exec")
    except Exception as e:
        context_variables["plots_status"] = "v2_failed" if is_v2 else "v1_failed"
        context_variables["last_executed_code"] = full_code
        context_variables["last_execution_output"] = f"{e}"
        context_variables["last_execution_exit_code"] = 1
        context_variables["last_executed_file"] = ""
        context_variables["next_agent"] = agent_name

        syntax_msg = str(e)
        if "expected an indented block after 'elif' statement" in syntax_msg:
            message = (
                "Error in python_coder_tool: generated code is not valid Python.\n"
                f"{e}\n"
                "The plotting block contains an incomplete elif branch. "
                "Use one stable if/elif/else chain and make every plot_type branch fully indented and self-contained."
            )
        else:
            message = f"Error in python_coder_tool: generated code is not valid Python.\n{e}"

        return ReplyResult(
            message=message,
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    python_code_model = PythonCode(code=full_code)

    script_path = os.path.join(project_folder, safe_name)
    try:
        with open(script_path, "w", encoding="utf-8") as f:
            f.write(python_code_model.code)
    except Exception as e:
        context_variables["plots_status"] = "v2_failed" if is_v2 else "v1_failed"
        _coder_retries += 1
        context_variables["coder_retries"] = _coder_retries
        if _coder_retries > _MAX_CODER_RETRIES:
            return ReplyResult(
                message=(
                    f"Error in python_coder_tool: AgentCoder exceeded {_MAX_CODER_RETRIES} retries. "
                    "Handing off to Human for manual intervention."
                ),
                target=AgentNameTarget("Human"),
                context_variables=context_variables,
            )
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in python_coder_tool: failed to store executed script.\n{e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    executor = LocalCommandLineCodeExecutor(timeout=600, work_dir=project_folder)
    code_block = CodeBlock(language="python", code=full_code)

    try:
        result = executor.execute_code_blocks([code_block])
    except Exception as e:
        context_variables["plots_status"] = "v2_failed" if is_v2 else "v1_failed"
        context_variables["last_executed_code"] = full_code
        context_variables["last_execution_output"] = f"{e}"
        context_variables["last_execution_exit_code"] = 1
        context_variables["last_executed_file"] = script_path
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message="Code execution failed due to an internal executor error:\n" + f"{e}\n",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    exit_code = result.exit_code
    output = result.output

    def _atomic_write_json(path: str, obj: dict) -> None:
        tmp_path = path + ".tmp"
        with open(tmp_path, "w", encoding="utf-8") as f:
            json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
        os.replace(tmp_path, path)

    try:
        plot_desc_by_id = {}
        plot_title_by_id = {}
        if isinstance(selected_specs, list):
            for s in selected_specs:
                if isinstance(s, dict):
                    pid = s.get("plot_id")
                    if isinstance(pid, str) and pid.strip():
                        pid_s = pid.strip()
                        title = s.get("title")
                        desc = s.get("description")
                        if isinstance(title, str) and title.strip():
                            plot_title_by_id[pid_s] = title.strip()
                        if isinstance(desc, str) and desc.strip():
                            plot_desc_by_id[pid_s] = desc.strip()

        if exit_code == 0 and expects_v1_plots:
            plots_dir_v1 = os.path.join(project_folder, "plots")
            os.makedirs(plots_dir_v1, exist_ok=True)

            png_files = []
            for fn in os.listdir(plots_dir_v1):
                if fn.lower().endswith(".png"):
                    png_files.append(fn)
            png_files.sort()

            produced_map = {}
            for fn in png_files:
                pid = _extract_plot_id_from_png_name(fn)
                if not pid:
                    context_variables["plots_status"] = "v1_failed"
                    raise ValueError(
                        f"PNG filename does not follow the required '<plot_id>_<slug>.png' contract: {fn}"
                    )
                produced_map.setdefault(pid, []).append(fn)

            if len(png_files) != 5:
                context_variables["plots_status"] = "v1_failed"
                raise ValueError(
                    f"Expected exactly 5 PNGs in '{plots_dir_v1}', but found {len(png_files)}: {png_files}"
                )

            dup = [pid for pid, files in produced_map.items() if len(files) > 1]
            if dup:
                context_variables["plots_status"] = "v1_failed"
                raise ValueError(
                    f"Duplicate plot_id prefixes detected in PNG filenames: {dup}"
                )

            expected_set = set(expected_plot_ids)
            produced_set = set(produced_map.keys())

            if produced_set != expected_set:
                context_variables["plots_status"] = "v1_failed"
                missing = sorted(list(expected_set - produced_set))
                extra = sorted(list(produced_set - expected_set))
                raise ValueError(
                    f"Produced plots do not match plot_selected. missing={missing}, extra={extra}"
                )

            plots_v1 = []
            for plot_id in expected_plot_ids:
                fn = produced_map[plot_id][0]
                rel_path = os.path.join("plots", fn).replace("\\", "/")
                description = plot_desc_by_id.get(plot_id, "") or plot_title_by_id.get(plot_id, "") or "Plotting V1 artifact."
                plots_v1.append(
                    {
                        "plot_id": plot_id,
                        "name": fn,
                        "path": rel_path,
                        "description": description,
                    }
                )

            context_variables["plots_v1"] = plots_v1
            context_variables["plots_status"] = "v1_ready"

            manifest_path = os.path.join(project_folder, "plots_v1_manifest.json")
            _atomic_write_json(manifest_path, {"plots_v1": plots_v1})

        if exit_code == 0 and is_v2:
            plots_dir_v2 = os.path.join(project_folder, "plots_v2")
            os.makedirs(plots_dir_v2, exist_ok=True)

            png_files_v2 = []
            for fn in os.listdir(plots_dir_v2):
                if fn.lower().endswith(".png"):
                    png_files_v2.append(fn)
            png_files_v2.sort()

            produced_map = {}
            for fn in png_files_v2:
                pid = _extract_plot_id_from_png_name(fn)
                if pid in plots_to_regenerate:
                    produced_map.setdefault(pid, []).append(fn)

            missing = [pid for pid in plots_to_regenerate if pid not in produced_map]
            if missing:
                context_variables["plots_status"] = "v2_failed"
                raise ValueError(f"Missing regenerated plots in plots_v2 for plot_ids: {missing}")

            dup = [pid for pid, files in produced_map.items() if len(files) > 1]
            if dup:
                context_variables["plots_status"] = "v2_failed"
                raise ValueError(f"Multiple regenerated PNGs found for plot_ids in plots_v2: {dup}")

            plots_v2 = []
            for pid in plots_to_regenerate:
                fn = produced_map[pid][0]
                rel_path = os.path.join("plots_v2", fn).replace("\\", "/")
                description = plot_desc_by_id.get(pid, "") or plot_title_by_id.get(pid, "") or "Plotting V2 artifact."
                plots_v2.append(
                    {
                        "plot_id": pid,
                        "name": fn,
                        "path": rel_path,
                        "description": description,
                    }
                )

            context_variables["plots_v2"] = plots_v2
            context_variables["plots_status"] = "v2_ready"

            manifest_path_v2 = os.path.join(project_folder, "plots_v2_manifest.json")
            _atomic_write_json(manifest_path_v2, {"plots_v2": plots_v2})

    except Exception as e:
        if is_v2 and context_variables.get("plots_status") not in {"v2_failed", "v2_ready"}:
            context_variables["plots_status"] = "v2_failed"
        if (not is_v2) and context_variables.get("plots_status") not in {"v1_failed", "v1_ready"}:
            context_variables["plots_status"] = "v1_failed"
        context_variables["next_agent"] = agent_name
        context_variables["last_executed_code"] = full_code
        context_variables["last_execution_output"] = output
        context_variables["last_execution_exit_code"] = exit_code
        context_variables["last_executed_file"] = script_path
        return ReplyResult(
            message=f"Plot artifact processing failed:\n{e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    context_variables["last_executed_code"] = full_code
    context_variables["last_execution_output"] = output
    context_variables["last_execution_exit_code"] = exit_code
    context_variables["last_executed_file"] = script_path

    if exit_code != 0:
        if is_v2 and context_variables.get("plots_status") != "v2_ready":
            context_variables["plots_status"] = "v2_failed"
        if (not is_v2) and context_variables.get("plots_status") != "v1_ready":
            context_variables["plots_status"] = "v1_failed"
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Code execution failed.\n{output}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    if is_v2:
        context_variables["next_agent"] = next_agent_v2
        return ReplyResult(
            message=f"Code executed successfully.\n{output}",
            target=AgentNameTarget(next_agent_v2),
            context_variables=context_variables,
        )

    context_variables["next_agent"] = next_agent_v1
    return ReplyResult(
        message=f"Code executed successfully.\n{output}",
        target=AgentNameTarget(next_agent_v1),
        context_variables=context_variables,
    )

# Tool — plot_validator

def plot_validator_tool(
    validation_notes: Annotated[str, "Short global validation notes about V1 plots."],
    context_variables: ContextVariables,
) -> ReplyResult:
    agent_name = "AgentPlotValidator"

    plan = context_variables.get("plan", {})
    routing = plan.get("routing", {}) if isinstance(plan, dict) else {}

    try:
        if not isinstance(validation_notes, str) or not validation_notes.strip():
            raise ValueError("validation_notes must be a non-empty string.")
    except ValueError as e:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in plot_validator_tool: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    plots_status = context_variables.get("plots_status", None)
    if plots_status != "v1_ready":
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=(
                "Error in plot_validator_tool: plots_status must be 'v1_ready' before validation. "
                f"Got '{plots_status}'."
            ),
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    try:
        plots_v1 = context_variables.get("plots_v1", None)
        if not isinstance(plots_v1, list) or len(plots_v1) == 0:
            raise ValueError("context_variables['plots_v1'] must be a non-empty list.")

        plotting = plan.get("plotting", {}) if isinstance(plan, dict) else {}
        vision_model = plotting.get("vision_model", "o3")
        if not isinstance(vision_model, str) or not vision_model.strip():
            vision_model = "o3"

        reviews = {}
        plots_to_regenerate = []
        regeneration_reasons = {}

        structural_tokens = [
            "missing axis",
            "wrong plot type",
            "empty plot",
            "no data",
            "insufficient data",
            "file missing",
            "corrupted",
            "unreadable",
            "legend missing",
            "axis label missing",
            "title missing",
        ]

        for item in plots_v1:
            if not isinstance(item, dict):
                raise ValueError("Each plot in plots_v1 must be a dict.")

            meta = validate_plot_artifact_meta(item)

            plot_id = meta.get("plot_id")
            plot_name = meta.get("name")
            plot_description = meta.get("description")
            plot_path = meta.get("path")

            expected_rel = f"plots/{plot_name}"
            if plot_path.replace("\\", "/") != expected_rel:
                raise ValueError(
                    f"Plot '{plot_id}' has inconsistent path. "
                    f"Expected '{expected_rel}', got '{plot_path}'."
                )

            if not isinstance(plot_description, str) or not plot_description.strip():
                raise ValueError(f"Plot '{plot_id}' must have a non-empty description.")

            res = evaluate_plot(
                plot_name=plot_name,
                plot_description=plot_description,
                model=vision_model,
                plot_path=plot_path,
            )

            reviews[plot_id] = res.model_dump()

            issues = res.issues if isinstance(res.issues, list) else []

            structural_issue = any(
                isinstance(issue, str) and any(
                    token in issue.lower()
                    for token in structural_tokens
                )
                for issue in issues
            )

            should_regenerate = False
            reason_parts = []

            if res.pass_fail is False:
                should_regenerate = True
                reason_parts.append("pass_fail=False")

            if res.severity == "high":
                should_regenerate = True
                reason_parts.append("severity=high")

            if res.severity == "medium" and structural_issue:
                should_regenerate = True
                reason_parts.append("severity=medium_with_structural_issue")

            if should_regenerate:
                plots_to_regenerate.append(plot_id)
                regeneration_reasons[plot_id] = {
                    "severity": res.severity,
                    "pass_fail": res.pass_fail,
                    "structural_issue": structural_issue,
                    "issues": issues,
                    "reason": ", ".join(reason_parts) if reason_parts else "regeneration_required",
                }

        if len(set(plots_to_regenerate)) != len(plots_to_regenerate):
            raise ValueError("plots_to_regenerate contains duplicates.")

    except ValueError as e:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in plot_validator_tool: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )
    except Exception as e:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in plot_validator_tool: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    context_variables["plot_reviews"] = reviews
    context_variables["plots_to_regenerate"] = plots_to_regenerate
    context_variables["plot_regeneration_reasons"] = regeneration_reasons
    context_variables["plot_validation_notes"] = validation_notes.strip()

    if plots_to_regenerate:
        context_variables["plots_status"] = "v1_failed"
        next_agent = routing.get("on_plot_validation_failed", "AgentCoder")
    else:
        context_variables["plots_status"] = "v1_validated"
        next_agent = routing.get("after_plot_validate", "Human")

    if not isinstance(next_agent, str) or not next_agent.strip():
        next_agent = "Human"

    context_variables["next_agent"] = next_agent

    message = {
        "validated_plots": list(reviews.keys()),
        "plots_to_regenerate": plots_to_regenerate,
        "regeneration_reasons": regeneration_reasons,
        "validation_notes": validation_notes.strip(),
    }

    return ReplyResult(
        message=json.dumps(message, indent=2, ensure_ascii=False),
        target=AgentNameTarget(next_agent),
        context_variables=context_variables,
    )

# Tool — latex_writer_tool

def latex_writer_tool(
    latex_sections: Annotated[str, "LaTeX body content (no preamble)."],
    context_variables: ContextVariables,
) -> ReplyResult:
    agent_name = "AgentLatexWriter"

    plan = context_variables.get("plan", {})
    routing = plan.get("routing", {}) if isinstance(plan, dict) else {}

    next_agent = routing.get("after_latex_write", None)
    if not isinstance(next_agent, str) or not next_agent.strip():
        next_agent = "AgentLatexCompiler"

    try:
        if not isinstance(latex_sections, str) or not latex_sections.strip():
            raise ValueError("latex_sections must be a non-empty string.")

        if "LATEX_TEMPLATE" not in globals() or not isinstance(globals().get("LATEX_TEMPLATE"), str):
            raise ValueError("LATEX_TEMPLATE is missing or invalid (must be a string).")

        if "{section_content}" not in globals().get("LATEX_TEMPLATE"):
            raise ValueError("LATEX_TEMPLATE must contain '{section_content}' placeholder.")

        bad_markers = [
            "\\documentclass",
            "\\usepackage",
            "\\begin{document}",
            "\\end{document}",
            "\\maketitle",
            "\\title{",
            "\\author{",
            "\\date{",
        ]
        for m in bad_markers:
            if m in latex_sections:
                raise ValueError(f"latex_sections must NOT include template/preamble markers (found '{m}').")

    except ValueError as e:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in latex_writer_tool: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    project_folder = os.environ.get("PROJECT_FOLDER", "ag2_project")
    os.makedirs(project_folder, exist_ok=True)

    sections_path = os.path.join(project_folder, "report_sections.tex")
    report_tex_path = os.path.join(project_folder, "report.tex")
    paper_tex_path = os.path.join(project_folder, "paper.tex")

    sections_s = latex_sections.strip() + "\n"

    try:
        with open(sections_path, "w", encoding="utf-8") as f:
            f.write(sections_s)

        report_tex = globals()["LATEX_TEMPLATE"].replace("{section_content}", sections_s)

        with open(report_tex_path, "w", encoding="utf-8") as f:
            f.write(report_tex)

        with open(paper_tex_path, "w", encoding="utf-8") as f:
            f.write(report_tex)

    except Exception as e:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in latex_writer_tool while writing files: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    context_variables["latex_sections"] = sections_s
    context_variables["report_sections_path"] = sections_path
    context_variables["report_tex_path"] = report_tex_path
    context_variables["paper_tex_path"] = paper_tex_path
    context_variables["latex_tex_path"] = paper_tex_path
    context_variables["next_agent"] = next_agent

    message = {
        "report_sections_path": sections_path,
        "report_tex_path": report_tex_path,
        "paper_tex_path": paper_tex_path,
        "latex_tex_path": paper_tex_path,
        "latex_sections_chars": len(sections_s),
    }

    return ReplyResult(
        message=json.dumps(message, indent=2, ensure_ascii=False),
        target=AgentNameTarget(next_agent),
        context_variables=context_variables,
    )

# Tool — latex_compiler_tool

def latex_compiler_tool(
    context_variables: ContextVariables,
    latex_engine: Annotated[
        Literal["tectonic", "pdflatex", "xelatex", "lualatex"],
        "LaTeX engine to use for compilation. Default: tectonic.",
    ] = "tectonic",
    passes: Annotated[int, "Number of compilation passes (1-3). Default: 2."] = 2,
) -> ReplyResult:
    agent_name = "AgentLatexCompiler"

    plan = context_variables.get("plan", {})
    routing = plan.get("routing", {}) if isinstance(plan, dict) else {}

    next_agent = routing.get("after_latex_compile", None)
    if not isinstance(next_agent, str) or not next_agent.strip():
        next_agent = "Human"

    try:
        if latex_engine not in {"tectonic", "pdflatex", "xelatex", "lualatex"}:
            raise ValueError("latex_engine must be one of: pdflatex, xelatex, lualatex.")
        if not isinstance(passes, int) or passes < 1 or passes > 3:
            raise ValueError("passes must be an int in [1, 3].")

        project_folder = os.environ.get("PROJECT_FOLDER", "ag2_project")
        if not isinstance(project_folder, str) or not project_folder.strip():
            raise ValueError("PROJECT_FOLDER env var missing/invalid.")

        latex_tex_path = context_variables.get("latex_tex_path", None)
        if not isinstance(latex_tex_path, str) or not latex_tex_path.strip():
            raise ValueError("Missing or invalid context_variables['latex_tex_path'].")

        if not os.path.isabs(latex_tex_path):
            tex_path_abs = os.path.join(project_folder, latex_tex_path)
        else:
            tex_path_abs = latex_tex_path

        tex_path_abs = os.path.abspath(tex_path_abs)

        project_abs = os.path.abspath(project_folder)
        if not tex_path_abs.startswith(project_abs):
            raise ValueError("latex_tex_path must be inside PROJECT_FOLDER.")

        if not os.path.exists(tex_path_abs):
            raise ValueError(f"LaTeX source not found at '{tex_path_abs}'.")

    except ValueError as e:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in latex_compiler_tool: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    context_variables["latex_tex_path_abs"] = tex_path_abs

    os.makedirs(project_folder, exist_ok=True)

    compile_timeout = 300
    executor = LocalCommandLineCodeExecutor(timeout=compile_timeout, work_dir=project_folder)

    tex_filename = os.path.basename(tex_path_abs)
    pdf_filename = os.path.splitext(tex_filename)[0] + ".pdf"
    log_filename = os.path.splitext(tex_filename)[0] + ".log"

    if latex_engine == "tectonic":
        cmd = f'tectonic "{tex_filename}"'
    else:
        cmd = (
            f"{latex_engine} "
            f"-interaction=nonstopmode "
            f"-halt-on-error "
            f"-file-line-error "
            f"\"{tex_filename}\""
        )

    output_all = []
    exit_code_final = 0

    try:
        for _ in range(passes):
            cb = CodeBlock(language="bash", code=cmd)
            res = executor.execute_code_blocks([cb])
            exit_code_final = res.exit_code
            output_all.append(res.output or "")
            if exit_code_final != 0:
                break

    except Exception as e:
        next_agent_fail = routing.get("on_latex_compile_failed", "AgentLatexWriter")
        context_variables["latex_status"] = "failed"
        context_variables["latex_compile_success"] = False
        context_variables["latex_compile_error"] = f"{e}"
        context_variables["next_agent"] = next_agent_fail
        return ReplyResult(
            message=f"Error in latex_compiler_tool: internal executor error: {e}",
            target=AgentNameTarget(next_agent_fail),
            context_variables=context_variables,
        )

    combined_output = "\n".join([x for x in output_all if x]).strip()

    pdf_path = os.path.join(project_folder, pdf_filename)
    log_path = os.path.join(project_folder, log_filename)

    if exit_code_final != 0:
        log_tail = ""
        try:
            if os.path.exists(log_path):
                with open(log_path, "r", encoding="utf-8", errors="ignore") as f:
                    txt = f.read()
                log_tail = txt[-4000:]
        except Exception:
            log_tail = ""

        next_agent_fail = routing.get("on_latex_compile_failed", "AgentLatexWriter")
        context_variables["latex_status"] = "failed"
        context_variables["latex_compile_success"] = False
        context_variables["latex_engine"] = latex_engine
        context_variables["latex_passes"] = passes
        context_variables["latex_compile_output"] = combined_output
        context_variables["latex_log_path"] = log_path if os.path.exists(log_path) else ""
        context_variables["latex_log_tail"] = log_tail
        context_variables["next_agent"] = next_agent_fail

        return ReplyResult(
            message=(
                "LaTeX compilation failed.\n"
                f"engine={latex_engine}, passes={passes}, exit_code={exit_code_final}\n"
                + (f"\n--- log tail ---\n{log_tail}\n" if log_tail else "")
            ),
            target=AgentNameTarget(next_agent_fail),
            context_variables=context_variables,
        )

    if not os.path.exists(pdf_path):
        next_agent_fail = routing.get("on_latex_compile_failed", "AgentLatexWriter")
        context_variables["latex_status"] = "failed"
        context_variables["latex_compile_success"] = False
        context_variables["latex_engine"] = latex_engine
        context_variables["latex_passes"] = passes
        context_variables["latex_compile_output"] = combined_output
        context_variables["next_agent"] = next_agent_fail
        return ReplyResult(
            message=f"Error in latex_compiler_tool: compilation succeeded but PDF not found at '{pdf_path}'.",
            target=AgentNameTarget(next_agent_fail),
            context_variables=context_variables,
        )

    context_variables["latex_status"] = "ready"
    context_variables["latex_compile_success"] = True
    context_variables["latex_engine"] = latex_engine
    context_variables["latex_passes"] = passes
    context_variables["latex_compile_output"] = combined_output
    context_variables["latex_pdf_path"] = pdf_path
    context_variables["report_pdf_path"] = pdf_path
    context_variables["next_agent"] = next_agent

    message = {
        "latex_status": "ready",
        "latex_pdf_path": pdf_path,
        "latex_tex_path": tex_path_abs,
        "latex_engine": latex_engine,
        "passes": passes,
    }

    return ReplyResult(
        message=json.dumps(message, indent=2, ensure_ascii=False),
        target=AgentNameTarget(next_agent),
        context_variables=context_variables,
    )

# Tool — handoff_to_human

def handoff_to_human_tool(
    message: Annotated[str, "Short explanation of why the pipeline needs human intervention."],
    context_variables: ContextVariables,
) -> ReplyResult:
    agent_name = "Human"

    try:
        if not isinstance(message, str) or not message.strip():
            raise ValueError("handoff message must be a non-empty string.")
    except ValueError as e:
        context_variables["next_agent"] = agent_name
        return ReplyResult(
            message=f"Error in handoff_to_human_tool: {e}",
            target=AgentNameTarget(agent_name),
            context_variables=context_variables,
        )

    msg = message.strip()
    context_variables["handoff_reason"] = msg
    context_variables["next_agent"] = agent_name

    return ReplyResult(
        message=msg,
        target=AgentNameTarget(agent_name),
        context_variables=context_variables,
    )


_required_tools = [
    "task_improver_tool",
    "planner_tool",
    "review_plan_tool",
    "material_retriever",
    "final_conclusion_tool",
    "plot_suggester_tool",
    "plot_selector_tool",
    "python_coder_tool",
    "plot_validator_tool",
    "latex_writer_tool",
    "latex_compiler_tool",
    "handoff_to_human_tool",
]

_missing = [
    t for t in _required_tools
    if t not in globals() or not callable(globals()[t])
]

if _missing:
    raise RuntimeError(f"Tools missing or not callable: {_missing}")

print(f"✓ Tools loaded OK ({len(_required_tools)}).")

✓ Tools loaded OK (12).


# 7. AGENTS

In [17]:
# Agent: Task Improver

task_improver_message = """
You are AgentTaskImprover.

Your task is to improve the provided main task for its specific goal.

CRITERIA
- Analyze the consistency of the main task.
- Check whether the main task is complete or lacks critical information.
- Remove redundancy only when doing so does not change meaning, constraints, or scope.
- Preserve the user's exact objective, constraints, thresholds, exclusions, and requested workflow elements.
- Preserve the user's original meaning as the highest-priority rule.
- Do not add, remove, broaden, narrow, reinterpret, or substitute any requirement from the user.
- Do not add examples, inferred exclusions, indirect compounds, safety expansions, standards, assumptions, or implied subgoals.
- Do not replace the user's wording with a scientifically broader or stricter version.
- Do not convert explicit requirements into softer summaries.
- Do not add new deliverables, output formats, ranking goals, extra scientific objectives, or concrete plot/report specifications that were not requested.
- If the user asked to include plotting, keep plotting in the improved task, but do not invent the plot type, variables, ranking logic, file format, or presentation format.
- If the task is already clear and complete, keep improved_task semantically equivalent to the user task and make only minimal wording edits.
- If information is missing, mention it in analysis/improvements, but do not silently fill it into improved_task unless it is strictly necessary to make the task executable and does not change user intent.
- In case of doubt, copy the user constraints exactly rather than paraphrasing them.
- Never change chemical/material constraints, numeric ranges, excluded elements, environmental context, or requested workflow steps.
- Preserve words like "stable", "include plotting", and explicit exclusions exactly when they are part of the user request.
- Do not add phrases like "or their compounds", "multiple candidates", "PNG", "PDF", "ranking", "best materials", or similar unless the user explicitly requested them.
- Focus only on improving wording for the exact task the user gave.

OUTPUT
- Always produce: analysis, improvements, improved_task.
- improved_task must remain semantically equivalent to the original user request.
- improved_task must not introduce any new requirement.
- improved_task must not remove any original requirement.
- If the original task is already good, improved_task should be a minimal clean rewrite or a near-verbatim preservation.

AGENTS IN THE TEAM
- AgentPlanner: creates a plan JSON from main_task_improved.
- AgentPlanReviewer: validates the plan and routes approve/revise/handoff.
- AgentMaterialsRetriever: queries Materials Project using whitelisted keys/fields.
- AgentAnalyzer: produces structured JSON analysis from mp_results.
- AgentPlotSuggester: proposes plot candidates from the analysis.
- AgentPlotSelector: selects the final plotting contract.
- AgentCoder: executes Python code to generate artifacts.
- AgentPlotValidator: validates generated plots.
- AgentLatexWriter: writes the report sections.
- AgentLatexCompiler: compiles the final report.
- Human: reformulates the query if needed.

TOOL USAGE
- You must always use task_improver_tool.
- Do not answer directly without using the tool.
"""

AgentTaskImprover = ConversableAgent(
    name="AgentTaskImprover",
    llm_config=llm_config,
    system_message=task_improver_message,
    human_input_mode="NEVER",
    functions=[task_improver_tool],
    function_map={"task_improver_tool": task_improver_tool},
)

## Agent: Planner

planner_message = """
You are AgentPlanner.

Your task is to create a clean multi-agent execution plan from context_variables["main_task_improved"].

TOOL-ONLY BEHAVIOR
- You must not write any free text.
- You must not explain, introduce, acknowledge, summarize, or comment on the task.
- You must not say things like "I have refined the task", "Here is the plan", "Now I will generate a plan", or similar.
- Your first output must be a planner_tool call.
- Your only output must be a planner_tool call.
- Do not produce any text before the tool call.
- Do not produce any text after the tool call.
- Any response that is not a direct planner_tool call is invalid.

RULES
- Use only information already present in context_variables.
- You must build a dict called plan and pass it to planner_tool as the 'plan' argument.
- The plan must be consistent with the toolchain and valid for planner_tool validation.
- Preserve the scientific meaning of context_variables["main_task_improved"].
- If the task already contains explicit numeric constraints or excluded elements, copy them into plan.retrieval.search_criteria without changing their values.
- Do not relax, tighten, replace, swap, or drop user constraints unless they are truly missing from the task.
- If planner_tool returns a validation error, fix only the fields mentioned in the error and keep all other valid plan content unchanged.
- Never convert a valid numeric range into a single number, null, or a different range.
- Never invent new retrieval criteria that are not supported by the task.

PLAN FORMAT
- schema_version: string
- main_task_improved: must match context_variables["main_task_improved"]
- fireground_signals: dict inferred from the user text (environment, risks, conditions, fuel)
- material_roles: dict (roles and why they matter)
- retrieval: dict with:
  - search_criteria: dict with canonical Materials Project proxy filters
  - fields: list[str]
  - sample_number: int
- plotting: dict with:
  - enabled: bool
  - n_candidates: int (must be 10)
  - n_selected: int (must be 5)
  - output_dir: string (use "plots")
  - plot_intents: list[str] (non-empty when enabled is True)
- documentation: dict with:
  - enabled: bool
  - template: string
  - output: string
- steps: list of dict steps with required ids:
  - retrieve
  - analyze
  - plot_suggest
  - plot_select
  - plot_generate_v1
  - plot_validate
  - plot_generate_v2
  - latex_write
  - latex_compile
- routing: dict with non-empty strings:
  - after_planner
  - on_approve
  - on_revise
  - on_handoff
  - after_plot_suggest
  - after_plot_select
  - after_code
  - after_plot_validate
  - on_plot_validation_failed
  - after_latex_write
  - after_latex_compile
  - on_latex_compile_failed

CANONICAL EXAMPLE
- retrieval.search_criteria example:
  {
    "energy_above_hull": [0.0, 0.05],
    "band_gap": [3.0, 20.0],
    "density": [3.0, 10.0],
    "exclude_elements": ["Pb", "Cd", "Hg"]
  }

RETRIEVAL CONSTRAINTS
- plan.retrieval.search_criteria must use only these keys:
  - band_gap, energy_above_hull, density, num_sites, k_voigt, g_voigt, elements, chemsys, exclude_elements
- Use the key "exclude_elements" exactly.
- Never use "excluded_elements".
- Do not use mongo operators ($lte, $gt, etc).
- Numeric filters must be ranges with both numbers set and ordered as [min, max]:
  - energy_above_hull: [0.0, 0.05]
  - band_gap: [3.0, 20.0]
  - density: [3.0, 10.0]
- Use list-style ranges for numeric filters.
- Do not output null for numeric ranges.
- Do not output a single number where a range is required.
- fields must be a subset of:
  - material_id, formula_pretty, band_gap, energy_above_hull, density, volume, symmetry, nsites, elements, chemsys, is_stable, bulk_modulus, shear_modulus

PLOTTING CONSTRAINTS
- plotting.enabled must be True.
- plotting.n_candidates must be 10.
- plotting.n_selected must be 5.
- plotting.output_dir must be "plots".
- plotting.plot_intents must be a non-empty list[str] when plotting.enabled is True.
- The Planner must explicitly decide that plotting is part of the workflow and include it in the plan.
- plot_intents must be data-grounded and must only depend on fields that will actually be available after retrieval and analysis.
- For direct material properties, only use fields included in plan.retrieval.fields.
- For derived aggregate views, only use analyzer-derived fields that are explicitly expected from the analysis stage:
  - stability_bins
  - band_gap_bins
- Do not mention fields in plot_intents that are absent from plan.retrieval.fields.
- Do not mention unsupported derived names.
- Do not use vague plot intents such as:
  - "plot relevant properties"
  - "plot material suitability"
  - "plot stability metrics"
  - "plot material performance"
- Every plot_intent must name concrete fields or explicit derived bins.
- Prefer intents in this style:
  - "scatter energy_above_hull vs band_gap"
  - "scatter density vs band_gap"
  - "histogram energy_above_hull"
  - "histogram density"
  - "bar stability_bins"
  - "bar band_gap_bins"
- If a field is not retrieved, do not mention it in plot_intents.
- Example: do not mention volume unless volume is included in plan.retrieval.fields.
- plot_intents should guide PlotSuggester toward plots that are directly buildable from retrieved fields or known analyzer-derived bins.

DOCUMENTATION CONSTRAINTS
- documentation.enabled must be True.
- documentation.template must be a non-empty string.
- documentation.output must be "PDF".
- latex_write step must use AgentLatexWriter.
- latex_compile step must use AgentLatexCompiler.

STEPS CONSTRAINTS
- Each step must include a non-empty 'agent'.
- Each step.agent must be exactly one of:
  AgentTaskImprover, AgentPlanner, AgentPlanReviewer, AgentMaterialsRetriever, AgentAnalyzer,
  AgentPlotSuggester, AgentPlotSelector, AgentPlotValidator, AgentCoder, AgentLatexWriter,
  AgentLatexCompiler, Human.
- Do not use Human for plot_validate. plot_validate must be handled by AgentPlotValidator.
- Required agent wiring (strict):
  - retrieve -> AgentMaterialsRetriever
  - analyze -> AgentAnalyzer
  - plot_suggest -> AgentPlotSuggester
  - plot_select -> AgentPlotSelector
  - plot_generate_v1 -> AgentCoder
  - plot_validate -> AgentPlotValidator
  - plot_generate_v2 -> AgentCoder
  - latex_write -> AgentLatexWriter
  - latex_compile -> AgentLatexCompiler

ROUTING CONSTRAINTS
- routing.after_planner must be "AgentPlanReviewer"
- routing.on_approve must be "AgentMaterialsRetriever"
- routing.on_revise must be "AgentTaskImprover"
- routing.on_handoff must be "Human"
- routing.after_plot_suggest must be "AgentPlotSelector"
- routing.after_plot_select must be "AgentCoder"
- routing.after_code must be "AgentPlotValidator"
- routing.after_plot_validate must be "AgentLatexWriter"
- routing.on_plot_validation_failed must be "AgentCoder"
- routing.after_latex_write must be "AgentLatexCompiler"
- routing.after_latex_compile must be "Human"
- routing.on_latex_compile_failed must be "AgentLatexWriter"

ROUTING
- After you build the plan, route to AgentPlanReviewer.

TOOL USAGE
- You must always use planner_tool.
- Call planner_tool with: plan, plan_description, success_criteria, plan_score.
- Do not answer directly without using the tool.
"""

AgentPlanner = ConversableAgent(
    name="AgentPlanner",
    llm_config=llm_config,
    system_message=planner_message,
    human_input_mode="NEVER",
    functions=[planner_tool],
    function_map={"planner_tool": planner_tool},
)

# Agent: Plan Reviewer

plan_reviewer_message = """
You are AgentPlanReviewer.

Your task is to review context_variables["plan"] and decide:
- approve -> go to retriever
- revise -> go back to task improver
- handoff -> go back to human

RULES
- Validate the plan shape and basic wiring.
- Keep review_notes short and concrete.
- If the plan is missing required keys or required steps defined in the plan, choose "revise".
- Do not change the plan here; only decide routing.

ROUTING CONSTRAINTS
- approve must route to "AgentMaterialsRetriever"
- revise must route to "AgentTaskImprover"
- handoff must route to "Human"

TOOL USAGE
- You must always use review_plan_tool.
- Do not answer directly without using the tool.
"""

AgentPlanReviewer = ConversableAgent(
    name="AgentPlanReviewer",
    llm_config=llm_config,
    system_message=plan_reviewer_message,
    human_input_mode="NEVER",
    functions=[review_plan_tool],
    function_map={"review_plan_tool": review_plan_tool},
)

# Agent: Materials Retriever

materials_retriever_message = """
You are AgentMaterialsRetriever.

Your task is to retrieve candidate materials from Materials Project
based strictly on the execution plan stored in context_variables["plan"].

WHAT YOU DO
- Read search_criteria, fields, and sample_number from plan["retrieval"].
- Call material_retriever exactly once.
- Do not invent, modify, or infer filters or fields.
- Store results in context_variables["mp_results"].
- Do not analyze or rank materials.

RULES
- Use exactly what the plan provides for retrieval.
- Use only whitelisted search_criteria keys and fields.
- Numeric filters must be tuples: (min, max) with both values set (no open-ended bounds).
- List filters must be lists of strings.
- No Mongo-style operators.

TOOL USAGE
- Always call material_retriever with no arguments.
- Do not pass search_criteria, fields, or sample_number explicitly.
- Do not answer directly.
"""

AgentMaterialsRetriever = ConversableAgent(
    name="AgentMaterialsRetriever",
    llm_config=llm_config,
    system_message=materials_retriever_message,
    human_input_mode="NEVER",
    functions=[material_retriever],
    function_map={"material_retriever": material_retriever},
)

# Agent: Analyzer

analyzer_message = """
You are AgentAnalyzer.

ROLE
- Build a data-grounded conclusion from Materials Project results.

INPUTS (from context_variables)
- context_variables["mp_results"]: list of entries returned by the retriever tool.
- context_variables.get("main_task_improved", ""): task intent.
- If main_task_improved is missing, proceed using mp_results only.

YOUR TASKS
1) Read mp_results and use only available fields (no assumptions).
2) Internally derive the structured information needed for the analyzer result:
   - band_gap: distribution signal only
   - energy_above_hull: stability bins and ranges
   - density: descriptive stats only
   - missing values: explicitly preserve missingness in the summary logic
3) Produce ONE short final_conclusion string for operational use.
   - Must be strictly data-grounded.
   - Must be fully consistent with the structured analysis that will be computed by the tool.
   - Use only computed statistics already supported by the data:
     - retrieved counts
     - missing counts
     - p10/p50/p90 ranges
     - stability bin counts
     - band gap pattern counts
   - Do not overstate compliance.
   - Do not describe the full retrieved set as satisfying a criterion if missing values exist for that field.
   - If energy_above_hull has missing values, make that limitation explicit or restrict the claim to entries with available energy_above_hull values.
   - If a field has missing values, do not write wording that implies full verified compliance for all retrieved materials.
   - Do not use wording like:
     - "all materials meet the criteria"
     - "X materials meet the criteria"
     - "the retrieved materials are suitable"
     - "the retrieved materials satisfy the requested constraints"
     - "Retrieved N materials with energy_above_hull, band_gap, and density within specified ranges"
     - "matching the requested criteria"
     unless that claim is directly supported by the structured summary without ambiguity and without missing values for the referenced fields.
   - Prefer faithful wording such as:
     - "Retrieved N materials."
     - "Among entries with available energy_above_hull values, M are near-stable (<= 0.05 eV/atom)."
     - "Band gaps are predominantly wide."
     - "Densities fall within the requested range."
     - "energy_above_hull is missing for X retrieved entries."
   - Keep the conclusion short, factual, and operational.
   - No subjective claims or recommendations.
   - No material re-ranking.
   - No re-listing materials.

OUTPUT FORMAT
- Do not answer directly.
- The tool will generate the final structured result containing:
  - analysis_summary (dict)
  - final_conclusion (short string)

TOOL USAGE
- Call final_conclusion_tool EXACTLY ONCE per turn. Never issue parallel or multiple tool calls in the same response.
- Pass ONLY:
  - final_conclusion
- Do not pass analysis_summary.
- Never wrap arguments inside a `parameters` key.
- The call must be:
  final_conclusion_tool(final_conclusion="...")
- After the tool call succeeds, output nothing further. Routing is handled automatically.
- Do not answer directly.
"""

AgentAnalyzer = ConversableAgent(
    name="AgentAnalyzer",
    llm_config=llm_config,
    system_message=analyzer_message,
    human_input_mode="NEVER",
    functions=[final_conclusion_tool],
    function_map={"final_conclusion_tool": final_conclusion_tool},
)

# Agent: Plot Suggester

plot_suggester_message = """
You are AgentPlotSuggester.

Your task is to propose exactly 10 visualization ideas based on:
- context_variables["mp_results"]
- context_variables.get("analysis_summary", {})
- context_variables.get("main_task_improved", "")
- context_variables["plan"]["retrieval"]["fields"]

RULES
- Propose exactly 10 candidates.
- Each candidate must be practical given the real available fields in the current run.
- Use plan.retrieval.fields as the primary contract for field availability.
- Prefer plots that can be generated directly from retrieved per-material fields in mp_results.
- Do not generate code here.
- Do not select the best plots here.
- Every candidate must follow PlotCandidate + PlotAxes constraints.
- Do not invent axis names.
- axes values must be valid field names that are actually available in the current run.
- Do not use a field in axes or data_requirements unless it is available in plan.retrieval.fields or clearly available in mp_results.
- Do not propose plots that require unavailable fields such as volume, bulk_modulus, shear_modulus, chemsys, nsites, or is_stable unless those fields are actually present in plan.retrieval.fields for this run.
- Do not use derived aggregate labels in this run.
- Do not use derived fields such as:
  - "stability_bins"
  - "band_gap_bins"
  - "count"
- Do not use similar-looking variants or aliases.
- If a field is not clearly available, do not use it.
- Prefer direct per-material plots only in this run.
- Keep the suggestions scientifically relevant to the task and the retrieved data.
- Do not invent scientific justifications that depend on unavailable properties.
- For this run, use only these robust direct numeric fields:
  - band_gap
  - density
  - energy_above_hull
- Do not use elements as an axis.
- Do not use formula_pretty, material_id, or is_stable as plotting axes.
- Do not use derived categorical summaries in this run.

FIELD CONSISTENCY RULES
- data_requirements must always be a non-empty list of non-empty strings.
- Every non-null axis field must appear in data_requirements.
- Do not mix direct raw-material fields and unsupported derived fields in the same plot.
- If axes.x, axes.y, axes.z, or axes.intensity is not null, it must be explicitly present in data_requirements.
- Before calling the tool, verify candidate by candidate that every non-null axis value is reflected in data_requirements under the exact same name.
- For this run, every data_requirements entry must be one of:
  - band_gap
  - density
  - energy_above_hull

CANONICAL DERIVED FIELD RULES
- Do not use derived aggregate fields in this run.
- Never use:
  - "stability_bins"
  - "band_gap_bins"
  - "count"
  - "counts"
  - "material_count"
  - "number_of_materials"
  - "materials_count"
  - "frequency"

AGGREGATION SAFETY RULES
- Derived aggregate plots are forbidden in this run.
- Do not use bar plots in this run.
- Do not use derived aggregate plots in this run.
- The total number of derived plots must be 0 out of 10.
- The total number of bar plots must be 0 out of 10.
- All 10 candidates must be direct per-material plots.
- The output should be easy for AgentCoder to implement directly from per-material rows.
- HARD LIMIT: 0 derived plots total.
- HARD LIMIT: 0 bar plots total.
- HARD LIMIT: all 10 candidates must be direct per-material plots.

DERIVED BAR CONTRACT
- Do not use derived bar plots in this run.

VALID BAR CHART EXAMPLE
- Do not use this pattern in this run.

FIRST-TRY VALIDITY RULE
- Your first output must already satisfy the tool contract.
- Do not rely on retries to fix:
  - invalid derived names
  - mismatches between axes and data_requirements
  - unsupported field usage
  - unnecessary aggregate plots
  - too many derived bar plots
  - wrong candidate count
  - invalid heatmap intensity fields
- Before calling the tool, check every candidate against the contract:
  - plot_id format is correct
  - axes fields are valid
  - every non-null axis field is included in data_requirements
  - no derived aliases are used
  - all plots are direct per-material plots
  - there are exactly 10 candidates
  - there are 0 derived plots
  - there are 0 bar plots
  - there are 0 heatmaps

CORRECTION RULE
- If a prior tool call failed, do not patch only the single failing candidate.
- Rebuild and resubmit a fully consistent list of exactly 10 candidates.
- Do not remove a candidate unless you also replace it in the same response.
- Never submit 9 candidates.
- Never make a local fix that breaks a global rule.
- On retry, re-check the full set against all constraints before calling the tool again.
- On retry, fall back to the safest candidate set:
  - only scatter_2d
  - only histogram
  - only scatter_3d
  - only direct per-material fields
  - no derived fields
  - no bar plots
  - no heatmaps

PLOT TYPE RULES
- plot_type must be one of: scatter_2d, histogram, scatter_3d.
- axes must always be a dict with keys: type, x, y, z, intensity.
- axes.type must be one of:
  - "2d" for scatter_2d
  - "hist" for histogram
  - "3d" for scatter_3d
- Unused axis keys must be null.
- Plot type constraints:
  - scatter_2d: requires axes.type="2d", axes.x and axes.y, and axes.z/intensity must be null
  - histogram: requires axes.type="hist", axes.x only, and axes.y/z/intensity must be null
  - scatter_3d: requires axes.type="3d", axes.x, axes.y, axes.z, and axes.intensity must be null
- Do not use:
  - bar
  - heatmap_2d

PREFERRED PLOTTING STRATEGY FOR THIS RUN
- Prioritize scatter_2d, histogram, and scatter_3d using directly retrieved numeric fields.
- Prefer relationships among:
  - band_gap
  - density
  - energy_above_hull
- Prefer distributions of:
  - band_gap
  - density
  - energy_above_hull
- Prefer plots that are likely to produce non-empty figures from the retrieved fields.
- Avoid categorical aggregation in this run.
- Prefer robust first-pass execution over decorative variety.
- Keep scientific usefulness high, but do not choose a fragile plot when a direct plot communicates the same idea safely.
- 3D plots are allowed and encouraged when they are directly supported by available numeric fields and are clearly useful.
- Do not use heatmaps in this run.
- Build 10 candidates using only safe combinations of the three numeric fields above.

ID RULES
- plot_id must be stable and use exactly this format:
  - "plot_1", "plot_2", ..., "plot_10"
- Never use ids like "P01", "plot_01", or any other variant.

QUALITY RULES
- No pie charts.
- No bar plots.
- No heatmaps.
- No custom axis keys.
- Do not use placeholder axis names.
- Do not propose plots that are likely to be empty because the required data are missing.
- Do not propose plots that depend on fields not retrieved in this run.
- Prefer the strongest, most interpretable plots for insulation/stability/density tradeoffs.
- Prefer simple plots that maximize first-pass execution success.
- Avoid near-duplicate candidates that communicate almost the same thing unless the difference is clearly scientifically useful.
- Use the 10 slots to cover:
  - pairwise 2D relationships
  - the three single-field histograms
  - multiple 3D views with different axis ordering when scientifically useful

FINAL SELF-CHECK BEFORE TOOL CALL
- Confirm there are exactly 10 candidates.
- Confirm every plot_id is unique and uses the exact required format.
- Confirm every candidate has: plot_id, title, description, plot_type, data_requirements, axes, rationale.
- Confirm every non-null axis field appears in data_requirements.
- Confirm there are 0 bar plots.
- Confirm there are 0 heatmaps.
- Confirm there are 0 derived plots.
- Confirm all 10 candidates are direct per-material plots.
- Confirm no candidate depends on unavailable fields.
- Confirm the full set is globally valid, not just locally patched.
- Confirm every candidate uses only:
  - band_gap
  - density
  - energy_above_hull

OUTPUT FORMAT
Each candidate must include:
- plot_id
- title
- description
- plot_type
- data_requirements
- axes
- rationale

OUTPUT EXAMPLE SHAPE
[
  {
    "plot_id": "plot_1",
    "title": "Band Gap vs Density",
    "description": "Scatter plot of band gap against density.",
    "plot_type": "scatter_2d",
    "data_requirements": ["band_gap", "density"],
    "axes": {
      "type": "2d",
      "x": "band_gap",
      "y": "density",
      "z": null,
      "intensity": null
    },
    "rationale": "Helps compare electronic and structural behavior."
  }
]

TOOL USAGE
- You must always use plot_suggester_tool.
- Call plot_suggester_tool with: candidates.
- Do not answer directly without using the tool.
"""

AgentPlotSuggester = ConversableAgent(
    name="AgentPlotSuggester",
    llm_config=llm_config,
    system_message=plot_suggester_message,
    human_input_mode="NEVER",
    functions=[plot_suggester_tool],
    function_map={"plot_suggester_tool": plot_suggester_tool},
)

# Agent: Plot Selector

plot_selector_message = """
You are AgentPlotSelector.

Your task is to select the top 5 plots from context_variables["plot_candidates"].

RULES
- You must select exactly 5 plots.
- You must select from the existing candidates only.
- Do not invent new plots.
- You must output only plot_id values, not full dicts.
- Keep selection_notes short and concrete.
- Do not write code here.
- The selected plots will become the plotting contract for AgentCoder.
- Prefer plots that are directly supported by available retrieved fields and are most likely to generate valid, interpretable figures.
- Prioritize plots that are most relevant to the task: stability, insulation-related electronic behavior, density, and their relationships.
- Prefer direct per-material plots over weak or speculative derived plots.
- Avoid selecting plots that depend on unavailable, weakly supported, or non-retrieved fields.
- Avoid selecting plots that are likely to be empty or hard to interpret.
- Avoid redundant selections when two plots communicate nearly the same information.
- Aim for a strong, balanced set of 5 plots with high scientific usefulness and clean visual interpretability.
- Prefer plots based on direct retrieved per-material fields over aggregate or derived plots when both communicate similar information.
- Be conservative with derived plots such as binned or count-based categorical plots unless they add clear value beyond the direct plots.
- If a simpler direct scatter or histogram plot is more robust than a derived bar, heatmap, or 3D plot, prefer the simpler direct plot.
- Do not select a derived plot just because it sounds more sophisticated.
- Prioritize plots that are easiest for the current AgentCoder to execute correctly.
- Prefer plots whose plot_type, axes, and data_requirements are clearly aligned.
- Avoid selecting plots that require ambiguous mapping between axes and data fields.
- Avoid selecting plots that require extra hidden transformations, aggregation, binning, or remapping not clearly specified in the candidate.
- Avoid selecting plots that depend on fields that may exist in theory but are not clearly safe for direct plotting by the current coder.
- If two plots are similarly useful scientifically, prefer the one with the more robust execution contract.
- If there is any doubt about execution robustness, prefer the safer plot.

SELECTION PRIORITIES
- First priority: data availability and direct support from retrieved fields.
- Second priority: robustness of execution with the current coder.
- Third priority: scientific relevance to the task.
- Fourth priority: visual clarity and interpretability.
- Fifth priority: useful diversity across the final 5 plots.
- When priorities are tied, prefer the more direct and robust plot.

OUTPUT FORMAT
- selected_plot_ids: list of 5 plot_id strings taken from plot_candidates
- selection_notes: short string explaining the choice

TOOL USAGE
- You must always use plot_selector_tool.
- Call plot_selector_tool with: selected_plot_ids, selection_notes.
- Do not answer directly without using the tool.
"""

AgentPlotSelector = ConversableAgent(
    name="AgentPlotSelector",
    llm_config=llm_config,
    system_message=plot_selector_message,
    human_input_mode="NEVER",
    functions=[plot_selector_tool],
    function_map={"plot_selector_tool": plot_selector_tool},
)

# Agent: Coder

coder_message = """
You are AgentCoder.

CRITICAL: NEVER assign or redefine `plot_selected` in your code. The runtime prelude already defines it as a Python list before your code runs. Use `plot_selected` directly without loading it from os.environ yourself. Any assignment to `plot_selected` is FORBIDDEN and will cause your code to be rejected.

ROLE
- Generate and execute Python code when needed by the pipeline.

DATA SOURCE
- The Analyzer persists data to materials_data.json in the project folder.
- Use materials_data.json as the single source of truth for plots and tables.
- Assume python_coder_tool sets the working directory to PROJECT_FOLDER. Use relative paths only.
- The execution runtime provides a variable named `plot_selected` loaded from the pipeline state.

TASKS
1) Always export a CSV summary table to "summary_table.csv" with columns:
   material_id, formula_pretty, band_gap, energy_above_hull, density
   - Accept both "energy_above_hull" and "e_above_hull".
2) Print a clean, readable table to stdout so results are visible at the end of execution.
   Use fixed-width columns with f"{value:<12}" style formatting to prevent concatenation:
   print(f"{row['material_id']:<20} {row['formula_pretty']:<30} {row['band_gap']:<15} {row['energy_above_hull']:<20} {row['density']:<15}")

PLOTTING INPUTS
- plot_selected: list of selected plot dicts from pipeline state
- plot_selection_notes: short string (optional)

LIBRARY RULES
- Use matplotlib only.
- Do NOT use seaborn.
- Do NOT import external visualization libraries.
- Standard library imports are allowed.
- If needed for 3D plotting, use mpl_toolkits.mplot3d.

DIRECTORY CONTRACT
- Plotting V1 outputs must be saved under: "plots/"
- Plotting V2 outputs (if requested later) must be saved under: "plots_v2/"

CRITICAL PATH RULE
- The working directory is already PROJECT_FOLDER.
- When saving V1 plots, always use:
  out_path = os.path.join("plots", f"{plot_id}_{slug}.png")
  plt.savefig(out_path)

- When saving V2 plots, always use:
  out_path = os.path.join("plots_v2", f"{plot_id}_{slug}.png")
  plt.savefig(out_path)

- Do not prefix with "ag2_project/".
- Do not use absolute paths.
- Do not use PROJECT_FOLDER inside savefig.
- After saving each plot, verify it exists with os.path.exists(out_path).
  If it does not exist, raise RuntimeError.

NAMING CONTRACT
- Every plot filename must start with the exact plot_id followed by an underscore.
- Format:
  "plots/<plot_id>_<slug>.png"
- Example:
  "plots/<plot_id>_band_gap_distribution.png"
- Never save plots without the plot_id prefix.

PLOT SELECTION CONTRACT
- plot_selected is the single source of truth for which plots must be generated.
- The runtime variable `plot_selected` already exists and must be used directly.
- Never redefine, overwrite, shadow, reconstruct, filter into, or reassign `plot_selected` inside the generated code.
- Never assign anything to `plot_selected`.
- Forbidden patterns include:
  - plot_selected = ...
  - plot_selected += ...
  - plot_selected[:] = ...
  - plot_selected = [...]
  - plot_selected = list(...)
  - plot_selected = json.loads(...)
- Never create a manual local replacement or hardcoded replica of the selected plot specs.
- Never generate a different subset than the one already provided in plot_selected.
- You must generate plots by iterating directly over plot_selected.
- Use exactly the plot_id value present in each spec.
- Never rename, normalize, reformat, or invent plot IDs.
- Forbidden examples:
  - P01, P02, P03
  - plot_01, plot_02
  - plot01, plot02
- Do not hardcode separate plot blocks unrelated to the runtime specs.
- The number of generated plots must be exactly len(plot_selected).

FOR EACH SELECTED SPEC
- Iterate directly over the runtime `plot_selected` variable.
- Do not create a second hardcoded list of plot specs.
- Do not copy `plot_selected` into a new list or variable for iteration.
- Always iterate directly over `plot_selected`.
- Do not redefine or reconstruct `plot_selected`.
- Do not manually rewrite titles, descriptions, axes, or plot ids for selected plots.
- Read from the current spec:
  - plot_id
  - title
  - description
  - plot_type
  - axes
  - data_requirements
- Use exactly the plot_id provided in the spec.
- Never rename, normalize, or modify plot_id values.
- Build the filename using that exact plot_id.
- Save to:
  os.path.join("plots", f"{plot_id}_{slug}.png")
- Never use any plot_id not present in plot_selected.
- Never manually recreate plot specifications outside the iteration loop.

NON-NEGOTIABLE
- If plot_selected is a list with 5 items, you must generate exactly 5 PNG files under "plots/".
- Exporting CSV alone is not sufficient.
- For every plot spec, you must:
  - create exactly one plotting branch based on plot_type
  - create exactly one figure for that plot
  - compute out_path using os.path.join("plots", ...) for V1 or os.path.join("plots_v2", ...) for V2
  - call plt.savefig(out_path)
  - verify os.path.exists(out_path)
  - call plt.close()
- Never skip saving a selected plot.
- Do not create multiple competing figure objects for the same selected plot.
- Do not create a generic figure first and then a second figure again inside a later branch.
- For each selected plot, create the figure only inside the matching plot_type branch.
- Never create a generic figure before deciding the plot_type branch.
- Never create one figure outside the branch and another figure again inside the same plot branch.
- For one selected plot, there must be exactly one figure creation path.

PLOT TYPES
- plot_type can be:
  - scatter_2d
  - histogram
  - bar
  - heatmap_2d
  - scatter_3d
- Use axes.x / axes.y / axes.z / axes.intensity as the plotting contract.
- Do not infer plotting semantics from field names alone.
- Do not build x/y/z/intensity arrays using ad hoc name checks unrelated to axes.
- Always derive plot series from the axes fields in the selected spec.
- If plot_type is not recognized or required axes/data are missing, generate a placeholder.

PLOT IMPLEMENTATION RULES
- Prefer simple, robust matplotlib code over complex code paths.
- Use a distinct branch per plot_type.
- Keep the per-type logic separate and minimal.
- Do not force histogram, scatter, heatmap, and 3D plots through one shared plotting path.
- Before plotting, load:
  - payload = json.load(f)
  - mp_results = payload.get("mp_results", [])
- mp_results is the only iterable list of materials.
- For each selected plot, derive clean series from mp_results using the exact fields specified in axes and data_requirements.
- Only use rows where all required fields for that selected plot are present and usable.
- Build plotting series from axes.x / axes.y / axes.z / axes.intensity explicitly.
- Do not use loose logic such as:
  - if "band_gap" in data_requirements: ...
  - if "density" in data_requirements: ...
  unless that logic is directly tied to axes.x / axes.y / axes.z / axes.intensity.
- For scatter_2d:
  - require valid numeric x and y arrays of equal non-zero length
  - create one normal 2D figure and plot x against y
- For histogram:
  - require a valid numeric x array with at least one value
  - histogram preparation must be completely independent from scatter_2d, scatter_3d, heatmap_2d, and bar
  - do not build or rely on y_data, z_data, or intensity_data for histogram plotting
  - do not reuse a shared scatter-style data preparation path for histograms
  - the histogram branch must consume one direct numeric series only
  - if the histogram data is missing or unusable, generate a placeholder
- For scatter_3d:
  - require valid numeric x, y, z arrays of equal non-zero length
  - create the figure and 3D axis only inside the scatter_3d branch
  - use one pattern only:
    fig = plt.figure()
    ax = fig.add_subplot(111, projection="3d")
  - do not create a generic 2D figure before entering the scatter_3d branch
  - do not create more than one figure for the same selected 3D plot
  - do not call plt.figure() earlier for that same plot and then create another figure again
  - do not mix 2D axis code with 3D axis code
  - do not use plt.gca(projection="3d")
  - the scatter_3d branch must be fully independent from the 2D branches
- For heatmap_2d:
  - require valid numeric x and y arrays of equal non-zero length
  - use axes.x as x, axes.y as y
  - do not substitute intensity in place of y
  - always call plt.figure() at the start of the heatmap_2d branch, before plt.hexbin()
  - if intensity is present and usable, pass it as the C parameter to plt.hexbin(); build intensity_data as a same-length array aligned with x_data and y_data by filtering all three fields together in a single pass
  - if the selected heatmap spec cannot be implemented safely from the available fields, generate a placeholder
- For bar:
  - only use a safe direct aggregation path
  - if the selected spec does not map directly to safe bar inputs, generate a placeholder rather than invent unsupported logic
- Never fabricate data.
- Never use random values to fill a plot.

DATA PREPARATION RULES
- Do not use one shared generic data-building path for all plot types.
- Prepare only the data needed by the current plot_type branch.
- scatter_2d must prepare only x and y.
- histogram must prepare only x.
- scatter_3d must prepare only x, y, and z.
- heatmap_2d must prepare only x, y, and intensity when semantically valid.
- bar must prepare only the values required by the selected bar spec.
- Do not populate unused arrays for a plot type.
- Do not carry scatter arrays into histogram logic.
- Do not carry histogram logic into scatter or 3D logic.

STABLE CONTROL FLOW RULES
- Use one single stable plotting branch structure inside the per-plot loop:
  if plot_type == "scatter_2d":
      ...
  elif plot_type == "histogram":
      ...
  elif plot_type == "bar":
      ...
  elif plot_type == "heatmap_2d":
      ...
  elif plot_type == "scatter_3d":
      ...
  else:
      ...
- Do not reorder, split, duplicate, or partially rebuild this branch structure across the script.
- Do not generate conditional plotting branches dynamically.
- Do not leave any if / elif / else branch empty.
- Every branch must contain a complete syntactically valid body.
- If a plot type cannot be implemented safely, complete that branch with a placeholder path instead of leaving the branch incomplete.
- Never emit an elif branch without a full indented block under it.
- Never place comments or blank placeholders as the only content under a branch.
- Never open a branch and then continue its body outside the branch indentation.
- Do not generate partial try/except or partial if/elif blocks.
- The plotting block must be valid Python before any runtime conditions are considered.
- Prefer repeating a small safe per-type block over generating a clever but fragile shared branch.

PLACEHOLDER RULE
- Placeholder figures are a fallback only when the selected plot cannot be generated safely from the available data.
- Do not silently prefer placeholders when a valid plot can be produced.
- If required data is missing, malformed, empty, length-mismatched, or the plot type is unsupported, generate a placeholder figure instead.
- Placeholder figures must still:
  - include the exact plot_id in the title
  - include the text "Insufficient data"
  - be saved with the required naming rule

PLOTTING V1
- If plot_selected is a list with selected items, generate exactly those selected plots.
- For each plot spec in plot_selected:
  - use plot_id as the filename prefix
  - save under "plots/" using os.path.join
  - use a title-based slug in the filename
  - never skip a plot
- Exactly len(plot_selected) PNG files must exist in "plots/" after execution.

PLOTTING V2
- If plotting V2 is requested later, do not assume variables from a previous script execution exist.
- Rebuild all required runtime data inside the new script, including loading materials_data.json and deriving any filtered/material series needed by the regenerated plots.
- V2 regeneration must only rely on data loaded in the current script and on the runtime plot_selected variable.
- Do not assume filtered_materials, x_data, y_data, z_data, or any prior variables already exist.
- Avoid duplicate regenerated artifacts for the same plot_id in plots_v2.

OUTPUT REGISTRY CONTRACT (V1)
- After generating V1 plots, write a manifest JSON file "plots_v1_manifest.json" in PROJECT_FOLDER:
  {
    "plots_v1": [
      {"plot_id": "<plot_id>", "name": "<filename>", "path": "plots/<filename>", "description": "<short_description>"}
    ]
  }
- Build the manifest by iterating over the same runtime `plot_selected` used for plot generation.
- Do not build the manifest from a separate hardcoded list.
- Each manifest entry must preserve the exact plot_id from plot_selected.
- The manifest must always include a non-empty "description" string for every plot entry.
- The filename recorded in "name" must be exactly the same filename that was used in savefig for that plot.
- The path recorded in "path" must be exactly "plots/<filename>" for that same saved file.
- Use one stable filename construction rule only:
  - filename = f"{plot_id}_{slug}.png"
- Do not switch between different naming styles across plotting and manifest creation.
- Do not use title-only filenames in the manifest.
- Do not omit the plot_id prefix in the manifest.
- Write the manifest atomically using a temporary file and os.replace.

OUTPUT REGISTRY CONTRACT (V2)
- If plotting V2 is requested later, write a manifest JSON file "plots_v2_manifest.json" in PROJECT_FOLDER:
  {
    "plots_v2": [
      {"plot_id": "<plot_id>", "name": "<filename>", "path": "plots_v2/<filename>", "description": "<short_description>"}
    ]
  }
- Build the manifest by iterating over the same runtime `plot_selected` used for plot generation, restricted to the plots being regenerated for V2.
- Do not build the manifest from a separate hardcoded list.
- Each manifest entry must preserve the exact plot_id from plot_selected.
- The manifest must always include a non-empty "description" string for every plot entry.
- The filename recorded in "name" must be exactly the same filename that was used in savefig for that plot.
- The path recorded in "path" must be exactly "plots_v2/<filename>" for that same saved file.
- Use one stable filename construction rule only:
  - filename = f"{plot_id}_{slug}.png"
- Do not switch between different naming styles across plotting and manifest creation.
- Do not use title-only filenames in the manifest.
- Do not omit the plot_id prefix in the manifest.
- Write the manifest atomically using a temporary file and os.replace.

MANIFEST GENERATION RULES
- Do not build manifest entries with nested f-strings or inline string expressions inside dict literals.
- Before appending each manifest entry, compute these variables explicitly inside the manifest loop:
  - plot_id
  - title
  - slug
  - filename
  - rel_path
  - description
- Use the same filename construction rule in manifest generation that was used during savefig for that same plot.
- The manifest must not invent or recompute a different naming convention.
- filename must always be:
  - f"{plot_id}_{slug}.png"
- rel_path must always be:
  - os.path.join("plots", filename) for V1
  - os.path.join("plots_v2", filename) for V2
- Do not compute filename from title alone.
- Do not omit plot_id from filename.
- Do not reuse slug from another loop or another scope.
- Do not leave filename, rel_path, or slug dependent on a previous plot iteration.
- Keep manifest construction simple, explicit, and valid Python.
- The filename used in the manifest must exactly match the filename used in savefig for that same plot.

CRITICAL IO RULE
- materials_data.json is a dict payload.
- Load it as:
  - payload = json.load(f)
  - mp_results = payload.get("mp_results", [])

TABLE RULES
- Build the table by iterating mp_results into rows.
- Missing fields must be written as empty values.
- Use "energy_above_hull" first and fall back to "e_above_hull".
- Print a compact table with aligned columns.
- Ensure visible separation between columns.
- The CSV and printed table must reflect the same row data.

ROBUSTNESS RULES
- Generated code must be syntactically complete.
- Do not leave unclosed parentheses, broken strings, incomplete assignments, invalid f-strings, or incomplete loops.
- Do not produce partial code snippets.
- Do not stop midway through a branch.
- Always produce one complete executable Python script.
- Keep control flow simple and stable.
- Make sure every if / elif / else block is valid Python.
- Make sure every plot_type branch is self-contained and syntactically complete.
- Make sure histogram, scatter_2d, scatter_3d, heatmap_2d, and bar each have their own valid branch logic.
- Do not share fragile partial code across incompatible plot types.
- Make sure every for / while / with / try block is complete and correctly indented.
- Make sure 3D plotting code is properly indented under its branch.
- Make sure every created figure is either saved or explicitly closed.
- Make sure manifest construction is valid Python and does not rely on nested quote patterns that break f-strings.
- Before returning code, verify that all strings, brackets, parentheses, and dict literals are closed correctly.
- Before returning code, verify that filename/path construction is done with precomputed variables instead of inline nested expressions inside dict fields.
- Before returning code, verify that every elif branch has a full indented body and that no plotting branch is empty.
- Before returning code, verify that the plotting block uses one stable if/elif/else chain and not fragmented conditional snippets.
- Before returning code, verify that histogram logic is isolated from scatter logic.
- Before returning code, verify that the scatter_3d branch creates its figure only inside the scatter_3d branch.
- Before returning code, verify that no selected plot creates more than one figure.
- Before returning code, verify that manifest filename construction exactly matches savefig filename construction.

FORBIDDEN CODE PATTERNS
- Do not assign anything to `plot_selected`.
- Forbidden:
  - plot_selected = ...
  - plot_selected += ...
  - plot_selected[:] = ...
  - plot_selected = [...]
  - plot_selected = list(...)
  - plot_selected = json.loads(...)
- Do not hardcode plot specs in a local list.
- Do not manually create separate selected-plot definitions unrelated to the runtime specs.
- Do not invent plot ids.
- Do not write the manifest from a different list than the one used to generate the plots.
- Do not fabricate missing intensity/count columns.
- Do not use random placeholder numeric data.
- Do not construct manifest entries using nested f-strings or inline expressions inside dict fields.
- Do not use patterns like: "name": f"{plot_spec["plot_id"]}_..."
- Do not use one filename rule for savefig and another filename rule for the manifest.
- Do not compute filename from title alone in the manifest.
- Do not reuse slug from a previous loop iteration.
- Do not return incomplete code.


AXIS LABEL UNITS — always use these exact units, never write "(units)" as a placeholder:
  band_gap → "Band Gap (eV)"
  density → "Density (g/cm³)"
  energy_above_hull → "Energy Above Hull (eV/atom)"
  formation_energy_per_atom → "Formation Energy (eV/atom)"
  volume → "Volume (Å³)"
  nsites → "Number of Sites"
  nelements → "Number of Elements"
  total_magnetization → "Total Magnetization (μB)"
If a field is not listed above, derive a reasonable human-readable label with units from the field name. Never use f"{field} (units)" or similar placeholders.
TOOL USAGE
- Always call python_coder_tool for code execution.
- Do not execute code directly.
"""

AgentCoder = ConversableAgent(
    name="AgentCoder",
    llm_config=llm_config,
    system_message=coder_message,
    human_input_mode="NEVER",
    functions=[python_coder_tool],
    function_map={"python_coder_tool": python_coder_tool},
)

# Agent: Plot Validator

plot_validator_message = """
You are AgentPlotValidator.

ROLE
- Validate plot artifacts already generated by the pipeline.
- You do not generate plots, edit plots, or explain results directly to the user.

HARD PRECONDITION
- context_variables["plots_status"] must be exactly "v1_ready" before validation.
- Only V1 plots are valid for plot validation in this stage.
- Do not validate V2 plots here.
- Do not validate when plots_status is:
  - v1_running
  - v1_failed
  - v2_running
  - v2_failed
  - v2_ready
  - or any other value different from "v1_ready"

RULES
- Validate only artifacts already stored in context_variables["plots_v1"].
- Do not regenerate plots here.
- Do not rewrite manifests here.
- Do not answer with a final user-facing explanation.
- Keep validation_notes short, concrete, and global.
- The tool is the only valid path for plot validation and routing.
- Never stop after a free-text summary when plots_status == "v1_ready".
- Never claim that plots passed validation unless plot_validator_tool has actually validated them.
- Never produce optimistic status text that conflicts with the actual pipeline state.
- Never discuss corrective actions, retries, or next steps in free text.

WHEN plots_status == "v1_ready"
- You MUST call plot_validator_tool.
- Provide:
  - validation_notes
- Do not reply in normal text instead of calling the tool.

WHEN plots_status != "v1_ready"
- Do not call plot_validator_tool.
- Return only a short structural status message.
- Do not perform visual validation.
- Do not discuss next steps.
- Do not claim success.
- Do not mention that plots are valid, ready, corrected, or complete unless plots_status is exactly "v1_ready" and the tool is called.

VALIDATION SCOPE
- The tool handles:
  - artifact contract checks
  - metadata validation
  - plot-level review
  - deciding which plot_ids must be regenerated
  - routing to the correct next agent

OUTPUT FORMAT
- If calling the tool:
  - send only validation_notes to plot_validator_tool
- If not calling the tool:
  - return only a short structural status message

TOOL USAGE
- If context_variables["plots_status"] == "v1_ready":
  - always use plot_validator_tool
- Otherwise:
  - do not use the tool
"""

AgentPlotValidator = ConversableAgent(
    name="AgentPlotValidator",
    llm_config=llm_config,
    system_message=plot_validator_message,
    human_input_mode="NEVER",
    functions=[plot_validator_tool],
    function_map={"plot_validator_tool": plot_validator_tool},
)

# Agent: Writer

writer_message = """
You are AgentLatexWriter.

CRITICAL FORMATTING RULE: Output ONLY valid LaTeX syntax. NEVER use Markdown formatting:
  - Do NOT use **bold** — use \\textbf{bold} instead
  - Do NOT use *italic* — use \\textit{italic} instead
  - Do NOT use - for bullet lists — use \\begin{itemize} \\item ... \\end{itemize} instead
  - Do NOT use # headers — use \\section{} and \\subsection{} instead
Any Markdown syntax in the LaTeX output will cause PDF compilation to fail.

ROLE
- Write the BODY content of a scientific paper in LaTeX.
- Your output will be inserted into a fixed LaTeX template (preamble already exists).

STRICT RULES (NON-NEGOTIABLE)
- DO NOT include \\documentclass.
- DO NOT include any \\usepackage.
- DO NOT include \\begin{document} or \\end{document}.
- DO NOT include \\title, \\author, \\date, or \\maketitle (the template already provides them).

MANDATORY CONTENT
Your LaTeX body MUST include, in this order:

1) \\begin{abstract} ... \\end{abstract}

2) \\section{Introduction}
3) \\section{Methods}
4) \\section{Results}
5) \\section{Discussion}
6) \\section{Conclusion}

FIGURES
- Include figures using:
  \\begin{figure}[H]
  \\centering
  \\includegraphics[width=0.8\\textwidth]{<filename>}
  \\caption{...}
  \\end{figure}

- Use ONLY PNG filenames that already exist in:
  - context_variables["plots_v1"] and/or context_variables["plots_v2"]
  Specifically use the field: item["name"]
- Use ONLY the filename (e.g., P01_xxx.png). Do NOT write plots/ or plots_v2/ in the includegraphics path.
- Do NOT invent images or filenames.

DATA RULES
- Do NOT invent data or numbers.
- Base the text strictly on:
  - context_variables["analysis_summary"]
  - context_variables["final_conclusion"]
  - context_variables["plots_v1"] / ["plots_v2"]
  - context_variables["plot_reviews"] (if available)

OUTPUT RULE
- You MUST call latex_writer_tool.
- Pass your LaTeX BODY as the parameter `latex_sections`.
- Do not answer directly.
"""

AgentLatexWriter = ConversableAgent(
    name="AgentLatexWriter",
    llm_config=llm_config,
    system_message=writer_message,
    human_input_mode="NEVER",
    functions=[latex_writer_tool],
    function_map={"latex_writer_tool": latex_writer_tool},
)

# Agent: LaTeX Compiler

compiler_message = """
You are AgentLatexCompiler.

ROLE
- Compile the LaTeX report into a PDF using the provided tool.

INPUTS (from context_variables)
- report_tex_path: path to report.tex (set by latex_writer_tool)
- report_tex_path_abs: optional absolute path (may exist)

RULES
- You must call latex_compiler_tool.
- Do not write LaTeX.
- Do not edit files directly.
- Use default latex_engine=pdflatex and passes=2 unless the plan explicitly requests otherwise.

FAILURE HANDLING
- If compilation fails, latex_compiler_tool will route to the failure handler agent (usually AgentLatexWriter).
- Do not loop manually.
- Do not answer directly outside the tool call.
"""

AgentLatexCompiler = ConversableAgent(
    name="AgentLatexCompiler",
    llm_config=llm_config,
    system_message=compiler_message,
    human_input_mode="NEVER",
    functions=[latex_compiler_tool],
    function_map={"latex_compiler_tool": latex_compiler_tool},
)

# Human Agent

Human = ConversableAgent(
    name="Human",
    llm_config=None,
    human_input_mode="ALWAYS",
)
print("✓ Agents created.")

✓ Agents created.


# 8. Pattern definition 


In [18]:

pattern = DefaultPattern(
    initial_agent=AgentTaskImprover,
    user_agent=Human,
    agents=[
        AgentTaskImprover,
        AgentPlanner,
        AgentPlanReviewer,
        AgentMaterialsRetriever,
        AgentAnalyzer,
        AgentPlotSuggester,
        AgentPlotSelector,
        AgentCoder,
        AgentPlotValidator,
        AgentLatexWriter,       
        AgentLatexCompiler,     
    ],
    context_variables=context_variables,
)

# 9. Group Chat

In [19]:


print("Hi! Type your query and press Enter.\n")
human_query = input("Query: ").strip()

if not human_query:
    raise ValueError("Query cannot be empty.")

context_variables["query"] = human_query

result, context, _ = initiate_group_chat(
    pattern=pattern,
    messages=[{"role": "user", "content": human_query}],
    max_rounds=50,
)

if context_variables.get("next_agent") == "Human":
    decision = input("Generate full LaTeX paper? (yes/no): ").strip().lower()

    if decision in {"yes", "y"}:
        context_variables["next_agent"] = "AgentLatexWriter"
        result, context, _ = initiate_group_chat(
            pattern=pattern,
            messages=[],
            max_rounds=20,
        )
    else:
        print("Workflow finished without paper generation.")

Hi! Type your query and press Enter.



Query:  Find stable inorganic materials suitable as high-voltage protective barrier coatings near power lines, with energy_above_hull between 0.0 and 0.05 eV/atom, band_gap between 3.0 and 20.0 eV, density between 3.0 and 10.0 g/cm3, excluding toxic elements such as Pb, Cd, and Hg. Include plotting in the workflow.


Human (to chat_manager):

Find stable inorganic materials suitable as high-voltage protective barrier coatings near power lines, with energy_above_hull between 0.0 and 0.05 eV/atom, band_gap between 3.0 and 20.0 eV, density between 3.0 and 10.0 g/cm3, excluding toxic elements such as Pb, Cd, and Hg. Include plotting in the workflow.

--------------------------------------------------------------------------------

Next speaker: AgentTaskImprover

AgentTaskImprover (to chat_manager):

***** Suggested tool call (call_9EVAAvpy7P24grAfmPrqZK0c): task_improver_tool *****
Arguments: 
{"analysis":"The task is generally well-structured and complete. It specifies the type of materials (stable inorganic) and their application (high-voltage protective barrier coatings near power lines), with clear parameters for energy_above_hull, band_gap, and density. It also excludes toxic elements, which is important for safety. Plotting is included in the workflow as requested. However, the task includes min

Retrieving SummaryDoc documents:   0%|          | 0/9336 [00:00<?, ?it/s]


>>>>>>>> EXECUTED FUNCTION material_retriever...
Call ID: call_pG6m9K4cww7isfGcOiKrqMEf
Input arguments: {'plan': None}
Output:
Retrieved 100 materials from Materials Project with search_criteria={'energy_above_hull': (0.0, 0.05), 'band_gap': (3.0, 20.0), 'density': (3.0, 10.0), 'exclude_elements': ['Pb', 'Cd', 'Hg']}.
***** ReplyResult transition (AgentMaterialsRetriever): AgentAnalyzer *****
_Group_Tool_Executor (to chat_manager):

***** Response from calling tool (call_pG6m9K4cww7isfGcOiKrqMEf) *****
Retrieved 100 materials from Materials Project with search_criteria={'energy_above_hull': (0.0, 0.05), 'band_gap': (3.0, 20.0), 'density': (3.0, 10.0), 'exclude_elements': ['Pb', 'Cd', 'Hg']}.
**********************************************************************

--------------------------------------------------------------------------------

Next speaker: AgentAnalyzer

AgentAnalyzer (to chat_manager):

***** Suggested tool call (call_mBjWFF37xvZGfa0yy9PhVV8y): final_conclusion_tool